In [1]:
# ============================================================
# NEW STACK NOTEBOOK — BLOCK 1
# LOAD + VALIDATE STACK-READY RESEARCH DATASET
#
# Purpose:
#   1) Load the research dataset and feature manifest
#   2) Normalize timestamp and sort rows
#   3) Identify targets, validity masks, and fold columns
#   4) Build the initial candidate feature registry
#   5) Perform strict sanity checks before any modeling
# ============================================================

import os
import json
import numpy as np
import pandas as pd

# -----------------------------
# 1) PATHS
# -----------------------------
DATA_DIR = "data_4h"
RESEARCH_DIR = os.path.join(DATA_DIR, "research")
OUTPUT_DIR = os.path.join(DATA_DIR, "stack_v2_outputs")

os.makedirs(OUTPUT_DIR, exist_ok=True)

DATA_PATH = os.path.join(RESEARCH_DIR, "stack_ready_research_4h.csv")
MANIFEST_PATH = os.path.join(RESEARCH_DIR, "stack_ready_feature_manifest_4h.json")

# -----------------------------
# 2) REQUIRED OBJECTS
# -----------------------------
REQUIRED_TARGETS = [
    "target_main_ret",
    "target_main_up",
    "target_main_dir3",
    "target_main_tradeable",
    "target_main_fwd_rv",
    "target_main_tail",
]

REQUIRED_VALIDITY_MASKS = [
    "valid_for_alpha_main",
    "valid_for_gate_main",
    "valid_for_vol_main",
    "valid_for_tail_main",
]

REQUIRED_MARKET_COLS = [
    "timestamp",
    "eth_close",
    "eth_open",
    "eth_high",
    "eth_low",
    "eth_volume",
]

# -----------------------------
# 3) LOAD DATASET
# -----------------------------
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Research dataset not found: {DATA_PATH}")

if not os.path.exists(MANIFEST_PATH):
    raise FileNotFoundError(f"Manifest not found: {MANIFEST_PATH}")

df = pd.read_csv(DATA_PATH, low_memory=False)

if "timestamp" not in df.columns:
    raise ValueError("Expected a 'timestamp' column in the research dataset.")

df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
if df["timestamp"].isna().any():
    bad_n = int(df["timestamp"].isna().sum())
    raise ValueError(f"Found {bad_n} invalid timestamps after parsing.")

df = df.sort_values("timestamp").reset_index(drop=True)

with open(MANIFEST_PATH, "r") as f:
    manifest = json.load(f)

# -----------------------------
# 4) BASIC COLUMN CHECKS
# -----------------------------
missing_market = [c for c in REQUIRED_MARKET_COLS if c not in df.columns]
missing_targets = [c for c in REQUIRED_TARGETS if c not in df.columns]
missing_validity = [c for c in REQUIRED_VALIDITY_MASKS if c not in df.columns]

if missing_market:
    raise ValueError(f"Missing required market columns: {missing_market}")

if missing_targets:
    raise ValueError(f"Missing required target columns: {missing_targets}")

if missing_validity:
    raise ValueError(f"Missing required validity masks: {missing_validity}")

# -----------------------------
# 5) FEATURE REGISTRY
# -----------------------------
manifest_numeric_features = manifest.get("candidate_features_numeric", [])
manifest_categorical_features = manifest.get("candidate_features_categorical", [])

# keep only features that actually exist
numeric_features_all = [c for c in manifest_numeric_features if c in df.columns]
categorical_features_all = [c for c in manifest_categorical_features if c in df.columns]

# fold columns
fold_cols = sorted([
    c for c in df.columns
    if c.startswith("wf_fold_") and c != "wf_fold_id"
])

if len(fold_cols) == 0:
    raise ValueError("No walk-forward fold columns were found.")

# -----------------------------
# 6) TARGET REGISTRY
# -----------------------------
TARGETS = {
    "alpha_binary": "target_main_up",
    "alpha_regression": "target_main_ret",
    "alpha_multiclass": "target_main_dir3",
    "gate_binary": "target_main_tradeable",
    "vol_regression": "target_main_fwd_rv",
    "tail_binary": "target_main_tail",
}

VALIDITY = {
    "alpha": "valid_for_alpha_main",
    "gate": "valid_for_gate_main",
    "vol": "valid_for_vol_main",
    "tail": "valid_for_tail_main",
}

# -----------------------------
# 7) QUALITY SNAPSHOT
# -----------------------------
feature_missing_df = pd.DataFrame({
    "feature": numeric_features_all,
    "missing_rate": [df[c].isna().mean() for c in numeric_features_all]
}).sort_values("missing_rate", ascending=False).reset_index(drop=True)

target_snapshot_rows = []
for k, col in TARGETS.items():
    target_snapshot_rows.append({
        "target_name": k,
        "column": col,
        "non_null": int(df[col].notna().sum()),
        "null": int(df[col].isna().sum()),
    })

target_snapshot_df = pd.DataFrame(target_snapshot_rows)

validity_snapshot_rows = []
for k, col in VALIDITY.items():
    validity_snapshot_rows.append({
        "mask_name": k,
        "column": col,
        "sum": float(df[col].sum()),
        "mean": float(df[col].mean()),
    })

validity_snapshot_df = pd.DataFrame(validity_snapshot_rows)

# -----------------------------
# 8) PRINT SUMMARY
# -----------------------------
print("=" * 80)
print("STACK V2 — BLOCK 1 SUMMARY")
print("=" * 80)
print("Dataset shape:", df.shape)
print("Start       :", df["timestamp"].min())
print("End         :", df["timestamp"].max())

print("\nManifest feature counts:")
print("  Numeric features found     :", len(numeric_features_all))
print("  Categorical features found :", len(categorical_features_all))

print("\nFold columns:")
print(fold_cols)

print("\nTarget availability:")
print(target_snapshot_df)

print("\nValidity mask summary:")
print(validity_snapshot_df)

print("\nTop 20 most-missing numeric features:")
print(feature_missing_df.head(20))

# -----------------------------
# 9) EARLY SANITY CHECKS
# -----------------------------
# Timestamp monotonicity
if not df["timestamp"].is_monotonic_increasing:
    raise ValueError("Timestamps are not strictly sorted after preprocessing.")

# At least some non-null rows per target
for target_name, target_col in TARGETS.items():
    if df[target_col].notna().sum() == 0:
        raise ValueError(f"Target {target_name} ({target_col}) has zero non-null rows.")

# At least some valid rows per mask
for mask_name, mask_col in VALIDITY.items():
    if df[mask_col].sum() == 0:
        raise ValueError(f"Validity mask {mask_name} ({mask_col}) has zero usable rows.")

# Candidate numeric feature count must be nontrivial
if len(numeric_features_all) < 20:
    raise ValueError("Too few numeric candidate features found. Something is wrong upstream.")

print("\nAll hard sanity checks passed.")

# -----------------------------
# 10) NOTEBOOK-LEVEL GLOBALS
# -----------------------------
# These are the base objects used by later blocks
NUMERIC_FEATURES_ALL = numeric_features_all
CATEGORICAL_FEATURES_ALL = categorical_features_all
FEATURE_MISSING_DF = feature_missing_df
FOLD_COLS = fold_cols
TARGET_REGISTRY = TARGETS
VALIDITY_REGISTRY = VALIDITY

print("\nNotebook globals created:")
print("  df")
print("  NUMERIC_FEATURES_ALL")
print("  CATEGORICAL_FEATURES_ALL")
print("  FEATURE_MISSING_DF")
print("  FOLD_COLS")
print("  TARGET_REGISTRY")
print("  VALIDITY_REGISTRY")

STACK V2 — BLOCK 1 SUMMARY
Dataset shape: (12001, 151)
Start       : 2020-10-21 16:00:00+00:00
End         : 2026-04-13 16:00:00+00:00

Manifest feature counts:
  Numeric features found     : 95
  Categorical features found : 5

Fold columns:
['wf_fold_0', 'wf_fold_1', 'wf_fold_2', 'wf_fold_3', 'wf_fold_4', 'wf_fold_5', 'wf_fold_6', 'wf_fold_7', 'wf_fold_8']

Target availability:
        target_name                 column  non_null  null
0      alpha_binary         target_main_up     11998     3
1  alpha_regression        target_main_ret     11998     3
2  alpha_multiclass       target_main_dir3     11998     3
3       gate_binary  target_main_tradeable     11998     3
4    vol_regression     target_main_fwd_rv     11998     3
5       tail_binary       target_main_tail     11998     3

Validity mask summary:
  mask_name                column      sum     mean
0     alpha  valid_for_alpha_main  11998.0  0.99975
1      gate   valid_for_gate_main  11998.0  0.99975
2       vol    valid_for

In [2]:
# ============================================================
# NEW STACK NOTEBOOK — BLOCK 2
# BUILD REFINED MODELING SAMPLES
#
# Purpose:
#   1) Build clean task-specific samples for alpha / gate / vol / tail
#   2) Apply feature-level missingness filtering
#   3) Apply row-level cleaning only after task setup
#   4) Keep fold columns and timestamp aligned
#   5) Store everything in a reusable MODELING_SAMPLES dict
# ============================================================

# -----------------------------
# 1) CONFIG
# -----------------------------
MAX_FEATURE_MISSING = 0.20   # drop columns if >20% missing
DROP_EXEC_DO_NOT_TRADE = True
DROP_WF_FOLD_ID = True

# Optional: do not use these as predictors for now
DROP_EXACT_ALWAYS = {
    "timestamp",
    "date",
    "wf_role",
    "wf_fold_id",
    "valid_for_alpha_main",
    "valid_for_gate_main",
    "valid_for_vol_main",
    "valid_for_tail_main",
}

if DROP_EXEC_DO_NOT_TRADE:
    DROP_EXACT_ALWAYS = DROP_EXACT_ALWAYS.union({"exec_do_not_trade_basic"})

# -----------------------------
# 2) CANDIDATE NUMERIC FEATURE SET
# -----------------------------
candidate_numeric = [
    c for c in NUMERIC_FEATURES_ALL
    if c in df.columns
    and c not in DROP_EXACT_ALWAYS
    and not c.startswith("wf_fold_")
    and not c.startswith("target_")
]

# keep only columns under the missingness threshold
candidate_numeric = [
    c for c in candidate_numeric
    if df[c].isna().mean() <= MAX_FEATURE_MISSING
]

print("=" * 80)
print("BLOCK 2 — REFINED MODELING SAMPLES")
print("=" * 80)
print("Candidate numeric features after global screening:", len(candidate_numeric))

# -----------------------------
# 3) TASK DEFINITIONS
# -----------------------------
TASK_DEFS = {
    "alpha": {
        "target_col": TARGET_REGISTRY["alpha_binary"],
        "validity_col": VALIDITY_REGISTRY["alpha"],
        "target_type": "classification"
    },
    "gate": {
        "target_col": TARGET_REGISTRY["gate_binary"],
        "validity_col": VALIDITY_REGISTRY["gate"],
        "target_type": "classification"
    },
    "vol": {
        "target_col": TARGET_REGISTRY["vol_regression"],
        "validity_col": VALIDITY_REGISTRY["vol"],
        "target_type": "regression"
    },
    "tail": {
        "target_col": TARGET_REGISTRY["tail_binary"],
        "validity_col": VALIDITY_REGISTRY["tail"],
        "target_type": "classification"
    }
}

# -----------------------------
# 4) HELPER
# -----------------------------
def build_task_sample(df_in, task_name, target_col, validity_col, target_type, feature_cols, fold_cols):
    task_df = df_in.copy()

    # keep only rows valid for the task
    task_df = task_df.loc[task_df[validity_col] == 1].copy()
    task_df = task_df.loc[task_df[target_col].notna()].copy()

    # select usable features that actually exist
    usable_features = [c for c in feature_cols if c in task_df.columns]

    # conservative row cleaning
    before_rows = len(task_df)
    task_df = task_df.dropna(subset=usable_features + [target_col]).copy()
    after_rows = len(task_df)

    # build X / y
    X_task = task_df[usable_features].copy()

    if target_type == "classification":
        y_task = task_df[target_col].astype(int).copy()
    else:
        y_task = task_df[target_col].astype(float).copy()

    # metadata
    meta_cols = ["timestamp", target_col] + [c for c in fold_cols if c in task_df.columns]
    meta_task = task_df[meta_cols].copy()

    summary = {
        "task_name": task_name,
        "target_col": target_col,
        "target_type": target_type,
        "n_rows_before_dropna": before_rows,
        "n_rows_after_dropna": after_rows,
        "rows_removed": before_rows - after_rows,
        "n_features": len(usable_features),
        "feature_cols": usable_features,
    }

    if target_type == "classification":
        summary["target_mean"] = float(y_task.mean())
        summary["target_counts"] = y_task.value_counts(dropna=False).sort_index().to_dict()
    else:
        summary["target_mean"] = float(y_task.mean())
        summary["target_std"] = float(y_task.std())

    return {
        "df": task_df,
        "X": X_task,
        "y": y_task,
        "meta": meta_task,
        "summary": summary
    }

# -----------------------------
# 5) BUILD ALL TASK SAMPLES
# -----------------------------
MODELING_SAMPLES = {}

for task_name, spec in TASK_DEFS.items():
    sample_obj = build_task_sample(
        df_in=df,
        task_name=task_name,
        target_col=spec["target_col"],
        validity_col=spec["validity_col"],
        target_type=spec["target_type"],
        feature_cols=candidate_numeric,
        fold_cols=FOLD_COLS
    )
    MODELING_SAMPLES[task_name] = sample_obj

# -----------------------------
# 6) PRINT SUMMARIES
# -----------------------------
summary_rows = []

for task_name, obj in MODELING_SAMPLES.items():
    s = obj["summary"]
    row = {
        "task": task_name,
        "target_col": s["target_col"],
        "target_type": s["target_type"],
        "n_rows_before_dropna": s["n_rows_before_dropna"],
        "n_rows_after_dropna": s["n_rows_after_dropna"],
        "rows_removed": s["rows_removed"],
        "n_features": s["n_features"],
        "target_mean": s["target_mean"],
    }

    if "target_std" in s:
        row["target_std"] = s["target_std"]

    summary_rows.append(row)

TASK_SUMMARY_DF = pd.DataFrame(summary_rows)

print("\n" + "=" * 80)
print("TASK SUMMARY")
print("=" * 80)
print(TASK_SUMMARY_DF)

# -----------------------------
# 7) ALPHA SAMPLE AS DEFAULT WORKING SAMPLE
# -----------------------------
# For the first next blocks we default to alpha,
# but all task samples are already built.
ALPHA_SAMPLE = MODELING_SAMPLES["alpha"]

alpha_df = ALPHA_SAMPLE["df"]
X_alpha = ALPHA_SAMPLE["X"]
y_alpha = ALPHA_SAMPLE["y"]
meta_alpha = ALPHA_SAMPLE["meta"]
alpha_feature_cols = ALPHA_SAMPLE["summary"]["feature_cols"]

print("\n" + "=" * 80)
print("DEFAULT WORKING SAMPLE = ALPHA")
print("=" * 80)
print("alpha_df shape :", alpha_df.shape)
print("X_alpha shape  :", X_alpha.shape)
print("y_alpha shape  :", y_alpha.shape)
print("Positive rate  :", y_alpha.mean())

print("\nAlpha target distribution:")
print(y_alpha.value_counts(dropna=False).sort_index())

print("\nFirst 30 alpha features:")
print(alpha_feature_cols[:30])

# -----------------------------
# 8) HARD SANITY CHECKS
# -----------------------------
if X_alpha.empty:
    raise ValueError("Alpha feature matrix is empty.")

if y_alpha.nunique() < 2:
    raise ValueError("Alpha target has fewer than 2 classes.")

for task_name, obj in MODELING_SAMPLES.items():
    if obj["X"].shape[0] == 0:
        raise ValueError(f"Task '{task_name}' ended with zero usable rows.")
    if obj["X"].shape[1] == 0:
        raise ValueError(f"Task '{task_name}' ended with zero usable features.")

print("\nAll task-level sanity checks passed.")

# -----------------------------
# 9) NOTEBOOK GLOBALS FOR NEXT BLOCKS
# -----------------------------
print("\nNotebook globals created:")
print("  MODELING_SAMPLES")
print("  TASK_SUMMARY_DF")
print("  ALPHA_SAMPLE")
print("  alpha_df")
print("  X_alpha")
print("  y_alpha")
print("  meta_alpha")
print("  alpha_feature_cols")

BLOCK 2 — REFINED MODELING SAMPLES
Candidate numeric features after global screening: 90

TASK SUMMARY
    task             target_col     target_type  n_rows_before_dropna  \
0  alpha         target_main_up  classification                 11998   
1   gate  target_main_tradeable  classification                 11998   
2    vol     target_main_fwd_rv      regression                 11974   
3   tail       target_main_tail  classification                 11974   

   n_rows_after_dropna  rows_removed  n_features  target_mean  target_std  
0                11297           701          90     0.512525         NaN  
1                11297           701          90     0.949898         NaN  
2                11297           677          90     0.022379    0.018207  
3                11297           677          90     0.548110         NaN  

DEFAULT WORKING SAMPLE = ALPHA
alpha_df shape : (11297, 151)
X_alpha shape  : (11297, 90)
y_alpha shape  : (11297,)
Positive rate  : 0.51252544923431


In [3]:
# ============================================================
# NEW STACK NOTEBOOK — BLOCK 3
# BUILD REFINED FOLD REGISTRY FROM CLEANED ALPHA SAMPLE
#
# Purpose:
#   1) Rebuild fold splits AFTER alpha sample cleaning
#   2) Keep only usable folds with non-empty train/valid/test
#   3) Verify chronological integrity
#   4) Build both label-index and positional-index mappings
#   5) Create ALPHA_FOLD_SPLITS for later modeling blocks
# ============================================================

# -----------------------------
# 1) DISCOVER FOLD COLUMNS IN CLEANED SAMPLE
# -----------------------------
alpha_fold_cols = sorted([c for c in alpha_df.columns if c in FOLD_COLS])

if len(alpha_fold_cols) == 0:
    raise ValueError("No fold columns found in alpha_df after cleaning.")

print("=" * 80)
print("BLOCK 3 — REFINED ALPHA FOLD REGISTRY")
print("=" * 80)
print("Fold columns found in alpha_df:")
print(alpha_fold_cols)

# -----------------------------
# 2) BUILD CLEANED FOLD SPLITS
# -----------------------------
ALPHA_FOLD_SPLITS = []

for fold_col in alpha_fold_cols:
    train_idx = alpha_df.index[alpha_df[fold_col] == "train"].tolist()
    valid_idx = alpha_df.index[alpha_df[fold_col] == "valid"].tolist()
    test_idx  = alpha_df.index[alpha_df[fold_col] == "test"].tolist()

    # keep only folds with all 3 segments present
    if len(train_idx) == 0 or len(valid_idx) == 0 or len(test_idx) == 0:
        print(f"Skipping {fold_col} because one segment is empty after cleaning.")
        continue

    split_obj = {
        "fold_name": fold_col,
        "train_idx": train_idx,
        "valid_idx": valid_idx,
        "test_idx": test_idx,
    }

    ALPHA_FOLD_SPLITS.append(split_obj)

if len(ALPHA_FOLD_SPLITS) == 0:
    raise ValueError("No usable folds remain after rebuilding cleaned alpha folds.")

print(f"\nUsable cleaned alpha folds: {len(ALPHA_FOLD_SPLITS)}")

# -----------------------------
# 3) CHRONOLOGY CHECKS
# -----------------------------
for split in ALPHA_FOLD_SPLITS:
    fold_name = split["fold_name"]

    train_start = alpha_df.loc[split["train_idx"], "timestamp"].min()
    train_end   = alpha_df.loc[split["train_idx"], "timestamp"].max()

    valid_start = alpha_df.loc[split["valid_idx"], "timestamp"].min()
    valid_end   = alpha_df.loc[split["valid_idx"], "timestamp"].max()

    test_start  = alpha_df.loc[split["test_idx"], "timestamp"].min()
    test_end    = alpha_df.loc[split["test_idx"], "timestamp"].max()

    if not (train_end < valid_start):
        raise ValueError(f"{fold_name} violates train < valid chronology")

    if not (valid_end < test_start):
        raise ValueError(f"{fold_name} violates valid < test chronology")

print("Chronology checks passed for all cleaned alpha folds.")

# -----------------------------
# 4) BUILD POSITIONAL INDEX VERSION
# -----------------------------
# X_alpha and y_alpha share alpha_df index order,
# but many sklearn workflows prefer integer positions.
row_pos_map_alpha = pd.Series(np.arange(len(alpha_df)), index=alpha_df.index)

for split in ALPHA_FOLD_SPLITS:
    split["train_pos"] = row_pos_map_alpha.loc[split["train_idx"]].tolist()
    split["valid_pos"] = row_pos_map_alpha.loc[split["valid_idx"]].tolist()
    split["test_pos"]  = row_pos_map_alpha.loc[split["test_idx"]].tolist()

print("Built both label-index and positional-index mappings.")

# -----------------------------
# 5) BUILD FOLD SUMMARY TABLE
# -----------------------------
fold_summary_rows = []

for split in ALPHA_FOLD_SPLITS:
    fold_name = split["fold_name"]

    train_y = y_alpha.loc[split["train_idx"]]
    valid_y = y_alpha.loc[split["valid_idx"]]
    test_y  = y_alpha.loc[split["test_idx"]]

    fold_summary_rows.append({
        "fold": fold_name,
        "n_train": len(train_y),
        "n_valid": len(valid_y),
        "n_test": len(test_y),

        "train_pos_rate": float(train_y.mean()),
        "valid_pos_rate": float(valid_y.mean()),
        "test_pos_rate": float(test_y.mean()),

        "train_start": alpha_df.loc[split["train_idx"], "timestamp"].min(),
        "train_end": alpha_df.loc[split["train_idx"], "timestamp"].max(),

        "valid_start": alpha_df.loc[split["valid_idx"], "timestamp"].min(),
        "valid_end": alpha_df.loc[split["valid_idx"], "timestamp"].max(),

        "test_start": alpha_df.loc[split["test_idx"], "timestamp"].min(),
        "test_end": alpha_df.loc[split["test_idx"], "timestamp"].max(),
    })

ALPHA_FOLD_SUMMARY_DF = pd.DataFrame(fold_summary_rows)

print("\n" + "=" * 80)
print("ALPHA FOLD SUMMARY")
print("=" * 80)
print(ALPHA_FOLD_SUMMARY_DF)

# -----------------------------
# 6) SANITY CHECKS
# -----------------------------
# ensure every fold has decent size
MIN_SEGMENT_ROWS = 200

for split in ALPHA_FOLD_SPLITS:
    if len(split["train_pos"]) < MIN_SEGMENT_ROWS:
        raise ValueError(f"{split['fold_name']} has too few train rows.")
    if len(split["valid_pos"]) < MIN_SEGMENT_ROWS:
        raise ValueError(f"{split['fold_name']} has too few valid rows.")
    if len(split["test_pos"]) < MIN_SEGMENT_ROWS:
        raise ValueError(f"{split['fold_name']} has too few test rows.")

print("\nAll fold-size checks passed.")

# -----------------------------
# 7) EXAMPLE FOLD INSPECTION
# -----------------------------
example_fold = ALPHA_FOLD_SPLITS[0]

print("\n" + "=" * 80)
print("EXAMPLE CLEANED ALPHA FOLD")
print("=" * 80)
print("Fold name :", example_fold["fold_name"])
print("Train rows:", len(example_fold["train_idx"]))
print("Valid rows:", len(example_fold["valid_idx"]))
print("Test rows :", len(example_fold["test_idx"]))

print("\nTrain period:",
      alpha_df.loc[example_fold["train_idx"], "timestamp"].min(),
      "->",
      alpha_df.loc[example_fold["train_idx"], "timestamp"].max())

print("Valid period:",
      alpha_df.loc[example_fold["valid_idx"], "timestamp"].min(),
      "->",
      alpha_df.loc[example_fold["valid_idx"], "timestamp"].max())

print("Test period :",
      alpha_df.loc[example_fold["test_idx"], "timestamp"].min(),
      "->",
      alpha_df.loc[example_fold["test_idx"], "timestamp"].max())

# -----------------------------
# 8) NOTEBOOK GLOBALS FOR NEXT BLOCKS
# -----------------------------
print("\nNotebook globals created:")
print("  ALPHA_FOLD_SPLITS")
print("  ALPHA_FOLD_SUMMARY_DF")
print("  row_pos_map_alpha")

BLOCK 3 — REFINED ALPHA FOLD REGISTRY
Fold columns found in alpha_df:
['wf_fold_0', 'wf_fold_1', 'wf_fold_2', 'wf_fold_3', 'wf_fold_4', 'wf_fold_5', 'wf_fold_6', 'wf_fold_7', 'wf_fold_8']

Usable cleaned alpha folds: 9
Chronology checks passed for all cleaned alpha folds.
Built both label-index and positional-index mappings.

ALPHA FOLD SUMMARY
        fold  n_train  n_valid  n_test  train_pos_rate  valid_pos_rate  \
0  wf_fold_0     4145      607     705        0.518938        0.522241   
1  wf_fold_1     4084      705     720        0.508080        0.490780   
2  wf_fold_2     4182      720     527        0.500000        0.452778   
3  wf_fold_3     4252      527     720        0.486359        0.535104   
4  wf_fold_4     4059      720     694        0.488298        0.541667   
5  wf_fold_5     4059      694     628        0.499630        0.501441   
6  wf_fold_6     4033      628     720        0.504587        0.519108   
7  wf_fold_7     4028      720     689        0.506455       

In [4]:
# ============================================================
# NEW STACK NOTEBOOK — BLOCK 4
# BUILD REFINED ALPHA EXPERT LIBRARY
#
# Purpose:
#   1) Partition alpha features into meaningful families
#   2) Define a principled set of alpha experts
#   3) Tie each expert to a feature family + model class
#   4) Validate expert usability before training
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier

print("=" * 80)
print("BLOCK 4 — REFINED ALPHA EXPERT LIBRARY")
print("=" * 80)

# -----------------------------
# 1) FEATURE FAMILY DEFINITIONS
# -----------------------------
all_alpha_features = alpha_feature_cols.copy()

# Core OHLCV / base market structure
base_price_features = [
    c for c in all_alpha_features
    if c.startswith("eth_") or c.startswith("btc_") or c.startswith("ethbtc_")
]

# Derivatives family
derivatives_features = [
    c for c in all_alpha_features
    if any(k in c for k in [
        "funding",
        "mark_",
        "oi_",
        "basis_"
    ])
]

# Regime family
regime_features = [
    c for c in all_alpha_features
    if any(k in c for k in [
        "_mom_",
        "_rv_",
        "range_",
        "regime_"
    ])
]

# Context / relative structure family
context_features = [
    c for c in all_alpha_features
    if any(k in c for k in [
        "ctx_",
        "btc_ret_",
        "btc_rv_",
        "ethbtc_ret_",
        "ethbtc_rv_"
    ])
]

# Execution/liquidity-style family
execution_features = [
    c for c in all_alpha_features
    if any(k in c for k in [
        "exec_",
        "volume",
        "quote_volume",
        "num_trades",
        "taker_buy"
    ])
]

# Faster features
fast_features = [
    c for c in all_alpha_features
    if any(k in c for k in [
        "_ret_1",
        "_ret_2",
        "_ret_6",
        "_logret_1",
        "_chg_1",
        "_chg_6",
        "range_"
    ])
]

# Slower features
slow_features = [
    c for c in all_alpha_features
    if any(k in c for k in [
        "_mom_12",
        "_mom_24",
        "_rv_24",
        "_rv_48",
        "_z_48",
        "regime_"
    ])
]

# Clean unique ordering helper
def unique_keep_order(seq):
    seen = set()
    out = []
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

base_price_features = unique_keep_order([c for c in base_price_features if c in X_alpha.columns])
derivatives_features = unique_keep_order([c for c in derivatives_features if c in X_alpha.columns])
regime_features = unique_keep_order([c for c in regime_features if c in X_alpha.columns])
context_features = unique_keep_order([c for c in context_features if c in X_alpha.columns])
execution_features = unique_keep_order([c for c in execution_features if c in X_alpha.columns])
fast_features = unique_keep_order([c for c in fast_features if c in X_alpha.columns])
slow_features = unique_keep_order([c for c in slow_features if c in X_alpha.columns])

FEATURE_FAMILIES = {
    "all": all_alpha_features,
    "base_price": base_price_features,
    "derivatives": derivatives_features,
    "regime": regime_features,
    "context": context_features,
    "execution": execution_features,
    "fast": fast_features,
    "slow": slow_features,
}

print("Feature family sizes:")
for k, v in FEATURE_FAMILIES.items():
    print(f"  {k:>12}: {len(v)}")

# -----------------------------
# 2) MINIMUM FAMILY SIZE CHECK
# -----------------------------
MIN_FEATURES_PER_EXPERT = 8

def ensure_min_features(feature_list, fallback_list, min_features=MIN_FEATURES_PER_EXPERT):
    if len(feature_list) >= min_features:
        return feature_list
    # fallback: augment with broad features but preserve uniqueness
    merged = unique_keep_order(feature_list + fallback_list)
    return merged[:max(min_features, len(merged))]

derivatives_features = ensure_min_features(derivatives_features, all_alpha_features)
regime_features = ensure_min_features(regime_features, all_alpha_features)
context_features = ensure_min_features(context_features, all_alpha_features)
execution_features = ensure_min_features(execution_features, all_alpha_features)
fast_features = ensure_min_features(fast_features, all_alpha_features)
slow_features = ensure_min_features(slow_features, all_alpha_features)

# refresh registry
FEATURE_FAMILIES["derivatives"] = derivatives_features
FEATURE_FAMILIES["regime"] = regime_features
FEATURE_FAMILIES["context"] = context_features
FEATURE_FAMILIES["execution"] = execution_features
FEATURE_FAMILIES["fast"] = fast_features
FEATURE_FAMILIES["slow"] = slow_features

print("\nFeature family sizes after fallback protection:")
for k, v in FEATURE_FAMILIES.items():
    print(f"  {k:>12}: {len(v)}")

# -----------------------------
# 3) EXPERT DEFINITIONS
# -----------------------------
ALPHA_EXPERT_LIBRARY = {
    # Strong baseline linear expert
    "broad_linear": {
        "features": FEATURE_FAMILIES["all"],
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=2500,
                C=1.0,
                random_state=42
            ))
        ]),
        "family": "all",
        "description": "Broad linear benchmark on all numeric alpha features"
    },

    # Strong broad nonlinear expert
    "broad_tree": {
        "features": FEATURE_FAMILIES["all"],
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=20,
                random_state=42,
                n_jobs=-1
            ))
        ]),
        "family": "all",
        "description": "Broad nonlinear tree expert on all numeric alpha features"
    },

    # Broad boosted expert
    "broad_boost": {
        "features": FEATURE_FAMILIES["all"],
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingClassifier(
                max_depth=5,
                learning_rate=0.05,
                max_iter=300,
                random_state=42
            ))
        ]),
        "family": "all",
        "description": "Broad boosted expert on all numeric alpha features"
    },

    # Derivatives-only expert
    "derivatives_tree": {
        "features": FEATURE_FAMILIES["derivatives"],
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(
                n_estimators=400,
                max_depth=6,
                min_samples_leaf=20,
                random_state=42,
                n_jobs=-1
            ))
        ]),
        "family": "derivatives",
        "description": "Expert focused on funding, mark, OI, basis-like features"
    },

    # Context / relative-value expert
    "context_tree": {
        "features": FEATURE_FAMILIES["context"],
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=350,
                max_depth=7,
                min_samples_leaf=20,
                random_state=42,
                n_jobs=-1
            ))
        ]),
        "family": "context",
        "description": "Expert focused on BTC / ETHBTC / relative context structure"
    },

    # Regime / slower state expert
    "regime_linear": {
        "features": FEATURE_FAMILIES["slow"],
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=2500,
                C=0.5,
                random_state=42
            ))
        ]),
        "family": "slow",
        "description": "Slower regime / vol / momentum linear expert"
    }
}

# -----------------------------
# 4) VALIDATE EXPERTS
# -----------------------------
expert_rows = []

for expert_name, spec in ALPHA_EXPERT_LIBRARY.items():
    feats = [c for c in spec["features"] if c in X_alpha.columns]
    ALPHA_EXPERT_LIBRARY[expert_name]["features"] = feats

    expert_rows.append({
        "expert_name": expert_name,
        "family": spec["family"],
        "n_features": len(feats),
        "description": spec["description"]
    })

ALPHA_EXPERT_SUMMARY_DF = pd.DataFrame(expert_rows).sort_values(
    ["family", "expert_name"]
).reset_index(drop=True)

print("\n" + "=" * 80)
print("ALPHA EXPERT SUMMARY")
print("=" * 80)
print(ALPHA_EXPERT_SUMMARY_DF)

# hard checks
bad_experts = ALPHA_EXPERT_SUMMARY_DF.loc[ALPHA_EXPERT_SUMMARY_DF["n_features"] < MIN_FEATURES_PER_EXPERT]
if len(bad_experts) > 0:
    raise ValueError(
        "Some experts have too few features after validation:\n"
        + bad_experts.to_string(index=False)
    )

print("\nAll alpha experts passed minimum feature-count checks.")

# -----------------------------
# 5) PRINT PREVIEW OF FEATURES PER EXPERT
# -----------------------------
for expert_name, spec in ALPHA_EXPERT_LIBRARY.items():
    print("\n" + "-" * 80)
    print(f"Expert: {expert_name}")
    print(f"Family: {spec['family']}")
    print(f"N features: {len(spec['features'])}")
    print("First 20 features:")
    print(spec["features"][:20])

# -----------------------------
# 6) NOTEBOOK GLOBALS FOR NEXT BLOCKS
# -----------------------------
print("\nNotebook globals created:")
print("  FEATURE_FAMILIES")
print("  ALPHA_EXPERT_LIBRARY")
print("  ALPHA_EXPERT_SUMMARY_DF")

BLOCK 4 — REFINED ALPHA EXPERT LIBRARY
Feature family sizes:
           all: 90
    base_price: 45
   derivatives: 21
        regime: 12
       context: 8
     execution: 18
          fast: 12
          slow: 9

Feature family sizes after fallback protection:
           all: 90
    base_price: 45
   derivatives: 21
        regime: 12
       context: 8
     execution: 18
          fast: 12
          slow: 9

ALPHA EXPERT SUMMARY
        expert_name       family  n_features  \
0       broad_boost          all          90   
1      broad_linear          all          90   
2        broad_tree          all          90   
3      context_tree      context           8   
4  derivatives_tree  derivatives          21   
5     regime_linear         slow           9   

                                         description  
0  Broad boosted expert on all numeric alpha feat...  
1  Broad linear benchmark on all numeric alpha fe...  
2  Broad nonlinear tree expert on all numeric alp...  
3  Expert f

In [5]:
# ============================================================
# NEW STACK NOTEBOOK — BLOCK 5
# TRAIN REFINED ALPHA EXPERTS OOF
#
# Purpose:
#   1) Train each alpha expert on cleaned alpha folds
#   2) Generate OOF validation probabilities
#   3) Generate forward test probabilities
#   4) Store fold-level metrics
#   5) Build alpha meta-training dataset
# ============================================================

from sklearn.metrics import roc_auc_score, log_loss, accuracy_score, brier_score_loss

print("=" * 80)
print("BLOCK 5 — TRAIN REFINED ALPHA EXPERTS OOF")
print("=" * 80)

# -----------------------------
# 1) OOF STORAGE
# -----------------------------
alpha_oof_df = alpha_df[["timestamp", TARGET_REGISTRY["alpha_binary"]]].copy()
alpha_oof_df = alpha_oof_df.rename(columns={TARGET_REGISTRY["alpha_binary"]: "y_true"})

for expert_name in ALPHA_EXPERT_LIBRARY.keys():
    alpha_oof_df[f"alpha_{expert_name}_valid"] = np.nan
    alpha_oof_df[f"alpha_{expert_name}_test"] = np.nan

alpha_fold_metric_rows = []

# -----------------------------
# 2) TRAIN EXPERTS FOLD BY FOLD
# -----------------------------
for split in ALPHA_FOLD_SPLITS:
    fold_name = split["fold_name"]

    train_pos = split["train_pos"]
    valid_pos = split["valid_pos"]
    test_pos = split["test_pos"]

    print("\n" + "-" * 80)
    print(f"Fold: {fold_name}")
    print(f"Train rows: {len(train_pos)} | Valid rows: {len(valid_pos)} | Test rows: {len(test_pos)}")

    for expert_name, spec in ALPHA_EXPERT_LIBRARY.items():
        feat_cols = spec["features"]
        model = spec["model"]

        X_train = X_alpha.iloc[train_pos][feat_cols]
        y_train = y_alpha.iloc[train_pos]

        X_valid = X_alpha.iloc[valid_pos][feat_cols]
        y_valid = y_alpha.iloc[valid_pos]

        X_test = X_alpha.iloc[test_pos][feat_cols]
        y_test = y_alpha.iloc[test_pos]

        model.fit(X_train, y_train)

        valid_prob = model.predict_proba(X_valid)[:, 1]
        test_prob = model.predict_proba(X_test)[:, 1]

        alpha_oof_df.loc[X_valid.index, f"alpha_{expert_name}_valid"] = valid_prob
        alpha_oof_df.loc[X_test.index, f"alpha_{expert_name}_test"] = test_prob

        alpha_fold_metric_rows.append({
            "fold": fold_name,
            "expert": expert_name,
            "family": spec["family"],
            "n_features": len(feat_cols),
            "n_train": len(y_train),
            "n_valid": len(y_valid),
            "n_test": len(y_test),

            "valid_auc": roc_auc_score(y_valid, valid_prob),
            "test_auc": roc_auc_score(y_test, test_prob),

            "valid_logloss": log_loss(y_valid, valid_prob),
            "test_logloss": log_loss(y_test, test_prob),

            "valid_brier": brier_score_loss(y_valid, valid_prob),
            "test_brier": brier_score_loss(y_test, test_prob),

            "valid_acc_05": accuracy_score(y_valid, (valid_prob >= 0.5).astype(int)),
            "test_acc_05": accuracy_score(y_test, (test_prob >= 0.5).astype(int)),
        })

        print(
            f"{expert_name:>18} | "
            f"valid_auc={roc_auc_score(y_valid, valid_prob):.4f} | "
            f"test_auc={roc_auc_score(y_test, test_prob):.4f} | "
            f"valid_ll={log_loss(y_valid, valid_prob):.4f}"
        )

# -----------------------------
# 3) METRICS TABLES
# -----------------------------
ALPHA_FOLD_METRICS_DF = pd.DataFrame(alpha_fold_metric_rows)

print("\n" + "=" * 80)
print("ALPHA FOLD METRICS (HEAD)")
print("=" * 80)
print(ALPHA_FOLD_METRICS_DF.head())

ALPHA_METRICS_SUMMARY_DF = (
    ALPHA_FOLD_METRICS_DF
    .groupby(["expert", "family"])[
        ["valid_auc", "test_auc", "valid_logloss", "test_logloss",
         "valid_brier", "test_brier", "valid_acc_05", "test_acc_05"]
    ]
    .agg(["mean", "std"])
    .sort_values(("valid_auc", "mean"), ascending=False)
)

print("\n" + "=" * 80)
print("ALPHA METRICS SUMMARY")
print("=" * 80)
print(ALPHA_METRICS_SUMMARY_DF)

# -----------------------------
# 4) OOF COVERAGE CHECK
# -----------------------------
print("\n" + "=" * 80)
print("ALPHA OOF COVERAGE CHECK")
print("=" * 80)

coverage_rows = []
for expert_name in ALPHA_EXPERT_LIBRARY.keys():
    valid_nonnull = alpha_oof_df[f"alpha_{expert_name}_valid"].notna().sum()
    test_nonnull = alpha_oof_df[f"alpha_{expert_name}_test"].notna().sum()

    coverage_rows.append({
        "expert": expert_name,
        "valid_nonnull": valid_nonnull,
        "test_nonnull": test_nonnull
    })

ALPHA_OOF_COVERAGE_DF = pd.DataFrame(coverage_rows)
print(ALPHA_OOF_COVERAGE_DF)

# -----------------------------
# 5) BUILD ALPHA META DATASET
# -----------------------------
alpha_meta_feature_cols = [
    f"alpha_{expert_name}_valid"
    for expert_name in ALPHA_EXPERT_LIBRARY.keys()
]

alpha_meta_df = alpha_oof_df[["timestamp", "y_true"] + alpha_meta_feature_cols].copy()
alpha_meta_df = alpha_meta_df.dropna(subset=alpha_meta_feature_cols).copy()

print("\n" + "=" * 80)
print("ALPHA META DATASET")
print("=" * 80)
print("alpha_meta_df shape:", alpha_meta_df.shape)
print("alpha_meta positive rate:", alpha_meta_df["y_true"].mean())
print("alpha meta features:", alpha_meta_feature_cols)

# -----------------------------
# 6) QUICK BASE COMPARISON TABLE
# -----------------------------
alpha_base_compare_rows = []

for expert_name in ALPHA_EXPERT_LIBRARY.keys():
    col = f"alpha_{expert_name}_valid"
    auc_ = roc_auc_score(alpha_meta_df["y_true"], alpha_meta_df[col])
    ll_ = log_loss(alpha_meta_df["y_true"], alpha_meta_df[col])
    brier_ = brier_score_loss(alpha_meta_df["y_true"], alpha_meta_df[col])
    acc_ = accuracy_score(alpha_meta_df["y_true"], (alpha_meta_df[col] >= 0.5).astype(int))

    alpha_base_compare_rows.append({
        "expert": expert_name,
        "auc": auc_,
        "logloss": ll_,
        "brier": brier_,
        "acc_05": acc_,
    })

ALPHA_BASE_COMPARE_DF = pd.DataFrame(alpha_base_compare_rows).sort_values("auc", ascending=False)

print("\n" + "=" * 80)
print("ALPHA BASE EXPERT COMPARISON (ON META SAMPLE)")
print("=" * 80)
print(ALPHA_BASE_COMPARE_DF)

# -----------------------------
# 7) SAVE ARTIFACTS
# -----------------------------
ALPHA_OOF_PATH = os.path.join(OUTPUT_DIR, "v2_alpha_expert_oof_predictions.csv")
ALPHA_FOLD_METRICS_PATH = os.path.join(OUTPUT_DIR, "v2_alpha_expert_fold_metrics.csv")
ALPHA_META_PATH = os.path.join(OUTPUT_DIR, "v2_alpha_meta_dataset.csv")
ALPHA_BASE_COMPARE_PATH = os.path.join(OUTPUT_DIR, "v2_alpha_base_compare.csv")

alpha_oof_df.to_csv(ALPHA_OOF_PATH, index=False)
ALPHA_FOLD_METRICS_DF.to_csv(ALPHA_FOLD_METRICS_PATH, index=False)
alpha_meta_df.to_csv(ALPHA_META_PATH, index=False)
ALPHA_BASE_COMPARE_DF.to_csv(ALPHA_BASE_COMPARE_PATH, index=False)

print("\nSaved artifacts:")
print(" ", ALPHA_OOF_PATH)
print(" ", ALPHA_FOLD_METRICS_PATH)
print(" ", ALPHA_META_PATH)
print(" ", ALPHA_BASE_COMPARE_PATH)

# -----------------------------
# 8) NOTEBOOK GLOBALS FOR NEXT BLOCKS
# -----------------------------
print("\nNotebook globals created:")
print("  alpha_oof_df")
print("  ALPHA_FOLD_METRICS_DF")
print("  ALPHA_METRICS_SUMMARY_DF")
print("  ALPHA_OOF_COVERAGE_DF")
print("  alpha_meta_df")
print("  alpha_meta_feature_cols")
print("  ALPHA_BASE_COMPARE_DF")

BLOCK 5 — TRAIN REFINED ALPHA EXPERTS OOF

--------------------------------------------------------------------------------
Fold: wf_fold_0
Train rows: 4145 | Valid rows: 607 | Test rows: 705
      broad_linear | valid_auc=0.7522 | test_auc=0.8122 | valid_ll=0.5973
        broad_tree | valid_auc=0.7683 | test_auc=0.8064 | valid_ll=0.6138
       broad_boost | valid_auc=0.7675 | test_auc=0.8094 | valid_ll=0.5857
  derivatives_tree | valid_auc=0.4644 | test_auc=0.5327 | valid_ll=0.7112
      context_tree | valid_auc=0.4588 | test_auc=0.5409 | valid_ll=0.7392
     regime_linear | valid_auc=0.4572 | test_auc=0.5303 | valid_ll=0.7013

--------------------------------------------------------------------------------
Fold: wf_fold_1
Train rows: 4084 | Valid rows: 705 | Test rows: 720
      broad_linear | valid_auc=0.8133 | test_auc=0.7885 | valid_ll=0.5133
        broad_tree | valid_auc=0.8076 | test_auc=0.7973 | valid_ll=0.5773
       broad_boost | valid_auc=0.8095 | test_auc=0.8056 | valid_ll

In [6]:
# ============================================================
# NEW STACK NOTEBOOK — BLOCK 6
# TRAIN REFINED ALPHA META-MODEL
#
# Purpose:
#   1) Train the refined alpha meta-model on expert OOF predictions
#   2) Evaluate stack vs base experts
#   3) Inspect meta-model coefficients
#   4) Save refined alpha stack artifacts
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score, brier_score_loss

print("=" * 80)
print("BLOCK 6 — TRAIN REFINED ALPHA META-MODEL")
print("=" * 80)

# -----------------------------
# 1) META INPUTS
# -----------------------------
X_alpha_meta = alpha_meta_df[alpha_meta_feature_cols].copy()
y_alpha_meta = alpha_meta_df["y_true"].astype(int).copy()

print("X_alpha_meta shape:", X_alpha_meta.shape)
print("y_alpha_meta shape:", y_alpha_meta.shape)
print("Positive rate:", y_alpha_meta.mean())

# -----------------------------
# 2) META MODEL
# -----------------------------
ALPHA_META_MODEL = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=3000,
        C=1.0,
        random_state=42
    ))
])

ALPHA_META_MODEL.fit(X_alpha_meta, y_alpha_meta)

alpha_stack_prob = ALPHA_META_MODEL.predict_proba(X_alpha_meta)[:, 1]
alpha_stack_pred_05 = (alpha_stack_prob >= 0.5).astype(int)

# -----------------------------
# 3) STACK METRICS
# -----------------------------
alpha_stack_auc = roc_auc_score(y_alpha_meta, alpha_stack_prob)
alpha_stack_ll = log_loss(y_alpha_meta, alpha_stack_prob)
alpha_stack_brier = brier_score_loss(y_alpha_meta, alpha_stack_prob)
alpha_stack_acc = accuracy_score(y_alpha_meta, alpha_stack_pred_05)

print("\n" + "=" * 80)
print("REFINED ALPHA STACK METRICS")
print("=" * 80)
print(f"AUC      : {alpha_stack_auc:.6f}")
print(f"LogLoss  : {alpha_stack_ll:.6f}")
print(f"Brier    : {alpha_stack_brier:.6f}")
print(f"Accuracy : {alpha_stack_acc:.6f}")

# -----------------------------
# 4) COEFFICIENT INSPECTION
# -----------------------------
alpha_meta_coef_df = pd.DataFrame({
    "meta_feature": alpha_meta_feature_cols,
    "coef": ALPHA_META_MODEL.named_steps["model"].coef_[0]
}).sort_values("coef", ascending=False).reset_index(drop=True)

alpha_meta_intercept = float(ALPHA_META_MODEL.named_steps["model"].intercept_[0])

print("\n" + "=" * 80)
print("ALPHA META COEFFICIENTS")
print("=" * 80)
print(alpha_meta_coef_df)
print("\nIntercept:", alpha_meta_intercept)

# -----------------------------
# 5) BUILD STACK OUTPUT FRAME
# -----------------------------
alpha_stack_df = alpha_meta_df.copy()
alpha_stack_df["alpha_stack_prob"] = alpha_stack_prob
alpha_stack_df["alpha_stack_pred_05"] = alpha_stack_pred_05

print("\n" + "=" * 80)
print("ALPHA STACK OUTPUT SAMPLE")
print("=" * 80)
print(alpha_stack_df.head())

# -----------------------------
# 6) BASE VS STACK COMPARISON
# -----------------------------
comparison_rows = []

for feat_col in alpha_meta_feature_cols:
    comparison_rows.append({
        "model": feat_col,
        "auc": roc_auc_score(y_alpha_meta, alpha_meta_df[feat_col]),
        "logloss": log_loss(y_alpha_meta, alpha_meta_df[feat_col]),
        "brier": brier_score_loss(y_alpha_meta, alpha_meta_df[feat_col]),
        "acc_05": accuracy_score(y_alpha_meta, (alpha_meta_df[feat_col] >= 0.5).astype(int))
    })

comparison_rows.append({
    "model": "alpha_stack_prob",
    "auc": alpha_stack_auc,
    "logloss": alpha_stack_ll,
    "brier": alpha_stack_brier,
    "acc_05": alpha_stack_acc
})

ALPHA_STACK_COMPARE_DF = pd.DataFrame(comparison_rows).sort_values("auc", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("BASE EXPERTS VS REFINED ALPHA STACK")
print("=" * 80)
print(ALPHA_STACK_COMPARE_DF)

# -----------------------------
# 7) SAVE ARTIFACTS
# -----------------------------
ALPHA_STACK_PATH = os.path.join(OUTPUT_DIR, "v2_alpha_stack_predictions.csv")
ALPHA_STACK_COMPARE_PATH = os.path.join(OUTPUT_DIR, "v2_alpha_stack_compare.csv")
ALPHA_META_COEF_PATH = os.path.join(OUTPUT_DIR, "v2_alpha_meta_coefficients.csv")

alpha_stack_df.to_csv(ALPHA_STACK_PATH, index=False)
ALPHA_STACK_COMPARE_DF.to_csv(ALPHA_STACK_COMPARE_PATH, index=False)
alpha_meta_coef_df.to_csv(ALPHA_META_COEF_PATH, index=False)

print("\nSaved artifacts:")
print(" ", ALPHA_STACK_PATH)
print(" ", ALPHA_STACK_COMPARE_PATH)
print(" ", ALPHA_META_COEF_PATH)

# -----------------------------
# 8) NOTEBOOK GLOBALS FOR NEXT BLOCKS
# -----------------------------
print("\nNotebook globals created:")
print("  ALPHA_META_MODEL")
print("  alpha_stack_df")
print("  alpha_stack_prob")
print("  alpha_meta_coef_df")
print("  ALPHA_STACK_COMPARE_DF")

BLOCK 6 — TRAIN REFINED ALPHA META-MODEL
X_alpha_meta shape: (6010, 6)
y_alpha_meta shape: (6010,)
Positive rate: 0.5119800332778702

REFINED ALPHA STACK METRICS
AUC      : 0.812469
LogLoss  : 0.514021
Brier    : 0.171496
Accuracy : 0.717471

ALPHA META COEFFICIENTS
                   meta_feature      coef
0        alpha_broad_tree_valid  1.242665
1       alpha_broad_boost_valid  0.167027
2      alpha_broad_linear_valid  0.139796
3     alpha_regime_linear_valid  0.001424
4      alpha_context_tree_valid -0.015216
5  alpha_derivatives_tree_valid -0.308290

Intercept: 0.06758016927781761

ALPHA STACK OUTPUT SAMPLE
                     timestamp  y_true  alpha_broad_linear_valid  \
4376 2022-10-21 00:00:00+00:00     1.0                  0.397549   
4377 2022-10-21 04:00:00+00:00     1.0                  0.889453   
4378 2022-10-21 08:00:00+00:00     1.0                  0.866868   
4379 2022-10-21 12:00:00+00:00     1.0                  0.514786   
4380 2022-10-21 16:00:00+00:00     0.0  

In [10]:
# ============================================================
# NEW STACK NOTEBOOK — BLOCK 7
# RESEARCH-GRADE META-LABEL GATE BUILDER (ADAPTIVE)
#
# Purpose:
#   1) Build a meta-label gate on top of the alpha stack
#   2) Select candidate-confidence threshold adaptively
#   3) Define gate label with cost- and vol-scaled hurdle
#   4) Build clean gate sample + folds without brittle failure
# ============================================================

print("=" * 80)
print("BLOCK 7 — RESEARCH-GRADE META-LABEL GATE BUILDER (ADAPTIVE)")
print("=" * 80)

# ------------------------------------------------
# 1) CONFIG
# ------------------------------------------------
TAU_GRID = np.round(np.arange(0.50, 0.951, 0.01), 2)

FIXED_COST_BP = 10.0
MIN_HURDLE_BP = 20.0
VOL_HURDLE_MULTIPLIER = 0.25

GATE_TARGET_COL = "target_gate_meta_v3"
GATE_VALIDITY_COL = "valid_for_gate_meta_v3"
CANDIDATE_COL = "gate_candidate_v3"

# Adaptive search regimes: strict -> moderate -> relaxed
SEARCH_REGIMES = [
    {
        "name": "strict",
        "min_coverage": 0.15,
        "max_coverage": 0.35,
        "min_pos_rate": 0.55,
        "max_pos_rate": 0.75,
        "min_selected": 800,
    },
    {
        "name": "moderate",
        "min_coverage": 0.10,
        "max_coverage": 0.45,
        "min_pos_rate": 0.50,
        "max_pos_rate": 0.80,
        "min_selected": 600,
    },
    {
        "name": "relaxed",
        "min_coverage": 0.07,
        "max_coverage": 0.55,
        "min_pos_rate": 0.45,
        "max_pos_rate": 0.85,
        "min_selected": 400,
    },
]

# ------------------------------------------------
# 2) LOAD PRIMARY ALPHA OUTPUTS INTO FULL DATAFRAME
# ------------------------------------------------
required_alpha_cols = ["timestamp", "alpha_stack_prob"] + alpha_meta_feature_cols
missing_alpha_cols = [c for c in required_alpha_cols if c not in alpha_stack_df.columns]
if missing_alpha_cols:
    raise ValueError(f"Missing required alpha-stack columns: {missing_alpha_cols}")

gate_base = df.copy()

alpha_merge_cols = ["timestamp", "alpha_stack_prob"] + alpha_meta_feature_cols
gate_base = gate_base.merge(
    alpha_stack_df[alpha_merge_cols].copy(),
    on="timestamp",
    how="inner"
).sort_values("timestamp").reset_index(drop=True)

if gate_base.empty:
    raise ValueError("Gate base dataset is empty after merging alpha stack outputs.")

# ------------------------------------------------
# 3) PRIMARY ALPHA SIDE / CONFIDENCE
# ------------------------------------------------
gate_base["alpha_side"] = np.where(gate_base["alpha_stack_prob"] >= 0.5, 1, -1)
gate_base["alpha_confidence"] = np.maximum(
    gate_base["alpha_stack_prob"],
    1.0 - gate_base["alpha_stack_prob"]
)
gate_base["alpha_edge"] = (gate_base["alpha_stack_prob"] - 0.5) * 2.0
gate_base["alpha_abs_edge"] = gate_base["alpha_edge"].abs()

if "target_main_ret" not in gate_base.columns:
    raise ValueError("Missing target_main_ret in gate_base.")
if "eth_rv_24" not in gate_base.columns:
    raise ValueError("Missing eth_rv_24 in gate_base.")

gate_base["realized_signed_ret"] = gate_base["alpha_side"] * gate_base["target_main_ret"]

fixed_cost_dec = FIXED_COST_BP / 10000.0
min_hurdle_dec = MIN_HURDLE_BP / 10000.0

rv_proxy = gate_base["eth_rv_24"].copy()
rv_proxy = rv_proxy.fillna(rv_proxy.median())

gate_base["gate_hurdle"] = np.maximum(
    np.maximum(fixed_cost_dec, min_hurdle_dec),
    VOL_HURDLE_MULTIPLIER * rv_proxy
)

# ------------------------------------------------
# 4) ADAPTIVE CONFIDENCE THRESHOLD SEARCH
# ------------------------------------------------
usable_for_search = gate_base["realized_signed_ret"].notna() & gate_base["alpha_confidence"].notna()
search_df = gate_base.loc[usable_for_search].copy()

all_search_rows = []
selected_regime_name = None
GATE_THRESHOLD_SEARCH_DF = None

for regime in SEARCH_REGIMES:
    search_rows = []

    for tau in TAU_GRID:
        sel = search_df["alpha_confidence"] >= tau
        coverage = float(sel.mean())

        if coverage < regime["min_coverage"] or coverage > regime["max_coverage"]:
            continue

        sel_df = search_df.loc[sel].copy()
        if len(sel_df) < regime["min_selected"]:
            continue

        candidate_label = (
            sel_df["realized_signed_ret"] > sel_df["gate_hurdle"]
        ).astype(int)

        pos_rate = float(candidate_label.mean())
        if pos_rate < regime["min_pos_rate"] or pos_rate > regime["max_pos_rate"]:
            continue

        net_signed_edge = sel_df["realized_signed_ret"] - fixed_cost_dec

        row = {
            "search_regime": regime["name"],
            "tau": float(tau),
            "coverage": coverage,
            "n_selected": int(len(sel_df)),
            "candidate_pos_rate": pos_rate,
            "mean_signed_ret": float(sel_df["realized_signed_ret"].mean()),
            "mean_net_edge_after_cost": float(net_signed_edge.mean()),
            "median_net_edge_after_cost": float(net_signed_edge.median()),
            "hit_rate_signed": float((sel_df["realized_signed_ret"] > 0).mean()),
            "hit_rate_above_hurdle": float(candidate_label.mean()),
        }
        search_rows.append(row)
        all_search_rows.append(row)

    if len(search_rows) > 0:
        GATE_THRESHOLD_SEARCH_DF = pd.DataFrame(search_rows).sort_values(
            ["mean_net_edge_after_cost", "hit_rate_above_hurdle", "coverage"],
            ascending=[False, False, True]
        ).reset_index(drop=True)
        selected_regime_name = regime["name"]
        break

if GATE_THRESHOLD_SEARCH_DF is None or GATE_THRESHOLD_SEARCH_DF.empty:
    ALL_GATE_THRESHOLD_SEARCH_DF = pd.DataFrame(all_search_rows)
    raise ValueError(
        "No admissible tau found even after adaptive relaxation. "
        "You should inspect alpha confidence distribution and hurdle severity."
    )

tau_star = float(GATE_THRESHOLD_SEARCH_DF.iloc[0]["tau"])

print("Selected search regime:", selected_regime_name)
print("Chosen tau* for candidate selection:", tau_star)
print("\nTop threshold-search rows:")
print(GATE_THRESHOLD_SEARCH_DF.head(10))

# store all evaluated rows across regimes if useful
ALL_GATE_THRESHOLD_SEARCH_DF = pd.DataFrame(all_search_rows)

# ------------------------------------------------
# 5) BUILD META-LABEL GATE TARGET
# ------------------------------------------------
gate_base[CANDIDATE_COL] = (gate_base["alpha_confidence"] >= tau_star).astype(int)

gate_base[GATE_TARGET_COL] = np.nan
candidate_mask = gate_base[CANDIDATE_COL] == 1

gate_base.loc[candidate_mask, GATE_TARGET_COL] = (
    gate_base.loc[candidate_mask, "realized_signed_ret"] >
    gate_base.loc[candidate_mask, "gate_hurdle"]
).astype(int)

gate_base[GATE_VALIDITY_COL] = (
    candidate_mask &
    gate_base["target_main_ret"].notna() &
    gate_base["alpha_stack_prob"].notna() &
    gate_base["eth_rv_24"].notna()
).astype(float)

print("\nNew gate target distribution (on candidate rows only):")
print(gate_base.loc[candidate_mask, GATE_TARGET_COL].value_counts(dropna=False).sort_index())

print("\nNew gate positive rate (candidate rows only):")
print(gate_base.loc[candidate_mask, GATE_TARGET_COL].mean())

print("\nCandidate coverage:")
print(candidate_mask.mean())

# ------------------------------------------------
# 6) BUILD GATE MODELING SAMPLE
# ------------------------------------------------
gate_df = gate_base.copy()
gate_df = gate_df.loc[gate_df[GATE_VALIDITY_COL] == 1].copy()
gate_df = gate_df.loc[gate_df[GATE_TARGET_COL].notna()].copy()

gate_feature_cols = candidate_numeric.copy()
gate_feature_cols = [c for c in gate_feature_cols if c in gate_df.columns]

alpha_gate_extra_features = [
    "alpha_stack_prob",
    "alpha_confidence",
    "alpha_edge",
    "alpha_abs_edge",
    "alpha_side",
] + alpha_meta_feature_cols

alpha_gate_extra_features = [c for c in alpha_gate_extra_features if c in gate_df.columns]

gate_feature_cols = list(dict.fromkeys(gate_feature_cols + alpha_gate_extra_features))

before_rows = len(gate_df)
gate_df = gate_df.dropna(subset=gate_feature_cols + [GATE_TARGET_COL]).copy()
after_rows = len(gate_df)

X_gate = gate_df[gate_feature_cols].copy()
y_gate = gate_df[GATE_TARGET_COL].astype(int).copy()

meta_gate = gate_df[["timestamp", GATE_TARGET_COL] + [c for c in FOLD_COLS if c in gate_df.columns]].copy()

print("\n" + "=" * 80)
print("GATE SAMPLE SUMMARY")
print("=" * 80)
print("Rows before dropna:", before_rows)
print("Rows after dropna :", after_rows)
print("Rows removed      :", before_rows - after_rows)
print("X_gate shape      :", X_gate.shape)
print("y_gate shape      :", y_gate.shape)
print("Positive rate     :", y_gate.mean())

print("\nGate target distribution:")
print(y_gate.value_counts(dropna=False).sort_index())

print("\nFirst 30 gate features:")
print(gate_feature_cols[:30])

# ------------------------------------------------
# 7) REBUILD GATE FOLDS AFTER CLEANING
# ------------------------------------------------
GATE_FOLD_SPLITS = []

gate_fold_cols = sorted([c for c in gate_df.columns if c in FOLD_COLS])

for fold_col in gate_fold_cols:
    train_idx = gate_df.index[gate_df[fold_col] == "train"].tolist()
    valid_idx = gate_df.index[gate_df[fold_col] == "valid"].tolist()
    test_idx  = gate_df.index[gate_df[fold_col] == "test"].tolist()

    if len(train_idx) == 0 or len(valid_idx) == 0 or len(test_idx) == 0:
        print(f"Skipping {fold_col} because one segment is empty after gate cleaning.")
        continue

    GATE_FOLD_SPLITS.append({
        "fold_name": fold_col,
        "train_idx": train_idx,
        "valid_idx": valid_idx,
        "test_idx": test_idx
    })

if len(GATE_FOLD_SPLITS) == 0:
    raise ValueError("No usable gate folds remain after rebuilding cleaned gate folds.")

for split in GATE_FOLD_SPLITS:
    fold_name = split["fold_name"]

    train_end = gate_df.loc[split["train_idx"], "timestamp"].max()
    valid_start = gate_df.loc[split["valid_idx"], "timestamp"].min()
    valid_end = gate_df.loc[split["valid_idx"], "timestamp"].max()
    test_start = gate_df.loc[split["test_idx"], "timestamp"].min()

    if not (train_end < valid_start):
        raise ValueError(f"{fold_name} violates train < valid chronology in gate sample")
    if not (valid_end < test_start):
        raise ValueError(f"{fold_name} violates valid < test chronology in gate sample")

row_pos_map_gate = pd.Series(np.arange(len(gate_df)), index=gate_df.index)

for split in GATE_FOLD_SPLITS:
    split["train_pos"] = row_pos_map_gate.loc[split["train_idx"]].tolist()
    split["valid_pos"] = row_pos_map_gate.loc[split["valid_idx"]].tolist()
    split["test_pos"]  = row_pos_map_gate.loc[split["test_idx"]].tolist()

# ------------------------------------------------
# 8) GATE FOLD SUMMARY
# ------------------------------------------------
gate_fold_rows = []

for split in GATE_FOLD_SPLITS:
    train_y = y_gate.loc[split["train_idx"]]
    valid_y = y_gate.loc[split["valid_idx"]]
    test_y  = y_gate.loc[split["test_idx"]]

    gate_fold_rows.append({
        "fold": split["fold_name"],
        "n_train": len(train_y),
        "n_valid": len(valid_y),
        "n_test": len(test_y),
        "train_pos_rate": float(train_y.mean()),
        "valid_pos_rate": float(valid_y.mean()),
        "test_pos_rate": float(test_y.mean()),
        "train_start": gate_df.loc[split["train_idx"], "timestamp"].min(),
        "train_end": gate_df.loc[split["train_idx"], "timestamp"].max(),
        "valid_start": gate_df.loc[split["valid_idx"], "timestamp"].min(),
        "valid_end": gate_df.loc[split["valid_idx"], "timestamp"].max(),
        "test_start": gate_df.loc[split["test_idx"], "timestamp"].min(),
        "test_end": gate_df.loc[split["test_idx"], "timestamp"].max(),
    })

GATE_FOLD_SUMMARY_DF = pd.DataFrame(gate_fold_rows)

print("\n" + "=" * 80)
print("GATE FOLD SUMMARY")
print("=" * 80)
print(GATE_FOLD_SUMMARY_DF)

# ------------------------------------------------
# 9) SANITY CHECKS
# ------------------------------------------------
if X_gate.empty:
    raise ValueError("X_gate is empty.")

if y_gate.nunique() < 2:
    raise ValueError("Gate target has fewer than 2 classes after redesign.")

if len(gate_feature_cols) < 10:
    raise ValueError("Too few gate features remain after filtering.")

print("\nAll gate sample sanity checks passed.")

# ------------------------------------------------
# 10) SAVE ARTIFACTS
# ------------------------------------------------
GATE_THRESHOLD_PATH = os.path.join(OUTPUT_DIR, "v2_gate_threshold_search.csv")
ALL_GATE_THRESHOLD_PATH = os.path.join(OUTPUT_DIR, "v2_gate_threshold_search_all_rows.csv")
GATE_SAMPLE_PATH = os.path.join(OUTPUT_DIR, "v2_gate_sample_preview.csv")
GATE_FOLD_SUMMARY_PATH = os.path.join(OUTPUT_DIR, "v2_gate_fold_summary.csv")

GATE_THRESHOLD_SEARCH_DF.to_csv(GATE_THRESHOLD_PATH, index=False)
ALL_GATE_THRESHOLD_SEARCH_DF.to_csv(ALL_GATE_THRESHOLD_PATH, index=False)
gate_df[[
    "timestamp", GATE_TARGET_COL, "alpha_stack_prob", "alpha_confidence",
    "alpha_side", "realized_signed_ret", "gate_hurdle"
]].to_csv(GATE_SAMPLE_PATH, index=False)
GATE_FOLD_SUMMARY_DF.to_csv(GATE_FOLD_SUMMARY_PATH, index=False)

print("\nSaved artifacts:")
print(" ", GATE_THRESHOLD_PATH)
print(" ", ALL_GATE_THRESHOLD_PATH)
print(" ", GATE_SAMPLE_PATH)
print(" ", GATE_FOLD_SUMMARY_PATH)

# ------------------------------------------------
# 11) NOTEBOOK GLOBALS FOR NEXT BLOCKS
# ------------------------------------------------
print("\nNotebook globals created:")
print("  tau_star")
print("  selected_regime_name")
print("  GATE_THRESHOLD_SEARCH_DF")
print("  ALL_GATE_THRESHOLD_SEARCH_DF")
print("  GATE_TARGET_COL")
print("  GATE_VALIDITY_COL")
print("  gate_df")
print("  X_gate")
print("  y_gate")
print("  meta_gate")
print("  gate_feature_cols")
print("  GATE_FOLD_SPLITS")
print("  GATE_FOLD_SUMMARY_DF")

BLOCK 7 — RESEARCH-GRADE META-LABEL GATE BUILDER (ADAPTIVE)
Selected search regime: relaxed
Chosen tau* for candidate selection: 0.75

Top threshold-search rows:
  search_regime   tau  coverage  n_selected  candidate_pos_rate  \
0       relaxed  0.75  0.511314        3073            0.849333   
1       relaxed  0.76  0.509318        3061            0.849396   
2       relaxed  0.74  0.511481        3074            0.849057   
3       relaxed  0.77  0.507820        3052            0.849607   
4       relaxed  0.73  0.513145        3084            0.847925   
5       relaxed  0.72  0.514143        3090            0.846602   
6       relaxed  0.71  0.515641        3099            0.845434   
7       relaxed  0.70  0.518303        3115            0.843018   
8       relaxed  0.69  0.521631        3135            0.840829   
9       relaxed  0.68  0.524126        3150            0.838730   

   mean_signed_ret  mean_net_edge_after_cost  median_net_edge_after_cost  \
0         0.020913      

In [11]:
# ============================================================
# NEW STACK NOTEBOOK — BLOCK 8
# TRAIN GATE MODEL V2 OOF
#
# Purpose:
#   1) Train the redesigned gate model on cleaned gate folds
#   2) Generate OOF validation and forward test probabilities
#   3) Compare trained gate vs naive alpha-confidence baseline
#   4) Save gate artifacts for later integration
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score, brier_score_loss

print("=" * 80)
print("BLOCK 8 — TRAIN GATE MODEL V2 OOF")
print("=" * 80)

# ------------------------------------------------
# 1) GATE MODEL
# ------------------------------------------------
# I use a regularized linear model first because:
# - sample is still modest
# - target is imbalanced
# - we want stable probabilities
GATE_MODEL = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=3000,
        C=0.5,
        class_weight="balanced",
        random_state=42
    ))
])

# ------------------------------------------------
# 2) OOF STORAGE
# ------------------------------------------------
gate_oof_df = gate_df[["timestamp", GATE_TARGET_COL, "alpha_confidence"]].copy()
gate_oof_df = gate_oof_df.rename(columns={GATE_TARGET_COL: "y_true"})

gate_oof_df["gate_prob_valid"] = np.nan
gate_oof_df["gate_prob_test"] = np.nan

gate_fold_metric_rows = []

# ------------------------------------------------
# 3) TRAIN FOLD BY FOLD
# ------------------------------------------------
for split in GATE_FOLD_SPLITS:
    fold_name = split["fold_name"]

    train_pos = split["train_pos"]
    valid_pos = split["valid_pos"]
    test_pos = split["test_pos"]

    X_train = X_gate.iloc[train_pos][gate_feature_cols]
    y_train = y_gate.iloc[train_pos]

    X_valid = X_gate.iloc[valid_pos][gate_feature_cols]
    y_valid = y_gate.iloc[valid_pos]

    X_test = X_gate.iloc[test_pos][gate_feature_cols]
    y_test = y_gate.iloc[test_pos]

    GATE_MODEL.fit(X_train, y_train)

    valid_prob = GATE_MODEL.predict_proba(X_valid)[:, 1]
    test_prob = GATE_MODEL.predict_proba(X_test)[:, 1]

    gate_oof_df.loc[X_valid.index, "gate_prob_valid"] = valid_prob
    gate_oof_df.loc[X_test.index, "gate_prob_test"] = test_prob

    # model metrics
    valid_auc = roc_auc_score(y_valid, valid_prob)
    test_auc = roc_auc_score(y_test, test_prob)

    valid_ll = log_loss(y_valid, valid_prob)
    test_ll = log_loss(y_test, test_prob)

    valid_brier = brier_score_loss(y_valid, valid_prob)
    test_brier = brier_score_loss(y_test, test_prob)

    valid_acc = accuracy_score(y_valid, (valid_prob >= 0.5).astype(int))
    test_acc = accuracy_score(y_test, (test_prob >= 0.5).astype(int))

    # naive baseline: alpha confidence itself
    baseline_valid = gate_df.iloc[valid_pos]["alpha_confidence"].values
    baseline_test = gate_df.iloc[test_pos]["alpha_confidence"].values

    base_valid_auc = roc_auc_score(y_valid, baseline_valid)
    base_test_auc = roc_auc_score(y_test, baseline_test)

    base_valid_ll = log_loss(y_valid, np.clip(baseline_valid, 1e-6, 1 - 1e-6))
    base_test_ll = log_loss(y_test, np.clip(baseline_test, 1e-6, 1 - 1e-6))

    gate_fold_metric_rows.append({
        "fold": fold_name,

        "valid_auc": valid_auc,
        "test_auc": test_auc,
        "valid_logloss": valid_ll,
        "test_logloss": test_ll,
        "valid_brier": valid_brier,
        "test_brier": test_brier,
        "valid_acc_05": valid_acc,
        "test_acc_05": test_acc,

        "baseline_valid_auc": base_valid_auc,
        "baseline_test_auc": base_test_auc,
        "baseline_valid_logloss": base_valid_ll,
        "baseline_test_logloss": base_test_ll,
    })

    print(
        f"{fold_name:>10} | "
        f"gate valid_auc={valid_auc:.4f} | gate test_auc={test_auc:.4f} | "
        f"base valid_auc={base_valid_auc:.4f} | base test_auc={base_test_auc:.4f}"
    )

# ------------------------------------------------
# 4) METRIC TABLES
# ------------------------------------------------
GATE_FOLD_METRICS_DF = pd.DataFrame(gate_fold_metric_rows)

print("\n" + "=" * 80)
print("GATE FOLD METRICS")
print("=" * 80)
print(GATE_FOLD_METRICS_DF)

GATE_METRICS_SUMMARY_DF = GATE_FOLD_METRICS_DF.agg({
    "valid_auc": ["mean", "std"],
    "test_auc": ["mean", "std"],
    "valid_logloss": ["mean", "std"],
    "test_logloss": ["mean", "std"],
    "valid_brier": ["mean", "std"],
    "test_brier": ["mean", "std"],
    "valid_acc_05": ["mean", "std"],
    "test_acc_05": ["mean", "std"],
    "baseline_valid_auc": ["mean", "std"],
    "baseline_test_auc": ["mean", "std"],
    "baseline_valid_logloss": ["mean", "std"],
    "baseline_test_logloss": ["mean", "std"],
})

print("\n" + "=" * 80)
print("GATE METRICS SUMMARY")
print("=" * 80)
print(GATE_METRICS_SUMMARY_DF)

# ------------------------------------------------
# 5) OOF COVERAGE CHECK
# ------------------------------------------------
print("\n" + "=" * 80)
print("GATE OOF COVERAGE CHECK")
print("=" * 80)
print("valid preds:", int(gate_oof_df["gate_prob_valid"].notna().sum()))
print("test preds :", int(gate_oof_df["gate_prob_test"].notna().sum()))

# ------------------------------------------------
# 6) META-READY GATE FRAME
# ------------------------------------------------
gate_meta_df = gate_oof_df[["timestamp", "y_true", "alpha_confidence", "gate_prob_valid"]].copy()
gate_meta_df = gate_meta_df.dropna(subset=["gate_prob_valid"]).copy()

print("\n" + "=" * 80)
print("GATE META DATASET")
print("=" * 80)
print("gate_meta_df shape:", gate_meta_df.shape)
print("positive rate:", gate_meta_df["y_true"].mean())

# ------------------------------------------------
# 7) DIRECT COMPARISON ON META SAMPLE
# ------------------------------------------------
gate_compare_rows = []

# baseline alpha confidence
base_prob = np.clip(gate_meta_df["alpha_confidence"].values, 1e-6, 1 - 1e-6)
y_true_gate = gate_meta_df["y_true"].astype(int).values

gate_compare_rows.append({
    "model": "alpha_confidence_baseline",
    "auc": roc_auc_score(y_true_gate, base_prob),
    "logloss": log_loss(y_true_gate, base_prob),
    "brier": brier_score_loss(y_true_gate, base_prob),
    "acc_05": accuracy_score(y_true_gate, (base_prob >= 0.5).astype(int)),
})

# trained gate
gate_prob = np.clip(gate_meta_df["gate_prob_valid"].values, 1e-6, 1 - 1e-6)
gate_compare_rows.append({
    "model": "trained_gate_v2",
    "auc": roc_auc_score(y_true_gate, gate_prob),
    "logloss": log_loss(y_true_gate, gate_prob),
    "brier": brier_score_loss(y_true_gate, gate_prob),
    "acc_05": accuracy_score(y_true_gate, (gate_prob >= 0.5).astype(int)),
})

GATE_COMPARE_DF = pd.DataFrame(gate_compare_rows).sort_values("auc", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("GATE MODEL VS BASELINE")
print("=" * 80)
print(GATE_COMPARE_DF)

# ------------------------------------------------
# 8) SAVE ARTIFACTS
# ------------------------------------------------
GATE_OOF_PATH = os.path.join(OUTPUT_DIR, "v2_gate_oof_predictions.csv")
GATE_FOLD_METRICS_PATH = os.path.join(OUTPUT_DIR, "v2_gate_fold_metrics.csv")
GATE_COMPARE_PATH = os.path.join(OUTPUT_DIR, "v2_gate_compare.csv")

gate_oof_df.to_csv(GATE_OOF_PATH, index=False)
GATE_FOLD_METRICS_DF.to_csv(GATE_FOLD_METRICS_PATH, index=False)
GATE_COMPARE_DF.to_csv(GATE_COMPARE_PATH, index=False)

print("\nSaved artifacts:")
print(" ", GATE_OOF_PATH)
print(" ", GATE_FOLD_METRICS_PATH)
print(" ", GATE_COMPARE_PATH)

# ------------------------------------------------
# 9) NOTEBOOK GLOBALS FOR NEXT BLOCKS
# ------------------------------------------------
print("\nNotebook globals created:")
print("  GATE_MODEL")
print("  gate_oof_df")
print("  GATE_FOLD_METRICS_DF")
print("  GATE_METRICS_SUMMARY_DF")
print("  gate_meta_df")
print("  GATE_COMPARE_DF")

BLOCK 8 — TRAIN GATE MODEL V2 OOF
 wf_fold_1 | gate valid_auc=0.5103 | gate test_auc=0.5654 | base valid_auc=0.5218 | base test_auc=0.5569
 wf_fold_2 | gate valid_auc=0.5421 | gate test_auc=0.4931 | base valid_auc=0.5569 | base test_auc=0.5666
 wf_fold_3 | gate valid_auc=0.4558 | gate test_auc=0.4789 | base valid_auc=0.5666 | base test_auc=0.5272
 wf_fold_4 | gate valid_auc=0.5336 | gate test_auc=0.5267 | base valid_auc=0.5272 | base test_auc=0.5534
 wf_fold_5 | gate valid_auc=0.4799 | gate test_auc=0.6066 | base valid_auc=0.5534 | base test_auc=0.5524
 wf_fold_6 | gate valid_auc=0.5868 | gate test_auc=0.5376 | base valid_auc=0.5524 | base test_auc=0.5367
 wf_fold_7 | gate valid_auc=0.5856 | gate test_auc=0.5748 | base valid_auc=0.5367 | base test_auc=0.5385

GATE FOLD METRICS
        fold  valid_auc  test_auc  valid_logloss  test_logloss  valid_brier  \
0  wf_fold_1   0.510264  0.565378       0.702254      0.518361     0.232249   
1  wf_fold_2   0.542090  0.493140       0.672369      

In [13]:
# ============================================================
# NEW STACK NOTEBOOK — BLOCK 9
# RESEARCH-GRADE VOLATILITY SAMPLE V2 + BASELINES + FOLDS
#
# Design choices:
#   1) Target = log( future realized volatility )
#   2) Keep HAR-style persistence features explicitly
#   3) Build naive / benchmark volatility forecasts inside sample
#   4) Prepare the dataset for QLIKE-aware model comparison later
#
# Required globals from previous blocks:
#   - df
#   - candidate_numeric
#   - FOLD_COLS
#   - TARGET_REGISTRY
#   - VALIDITY_REGISTRY
# ============================================================

print("=" * 80)
print("BLOCK 9 — RESEARCH-GRADE VOLATILITY SAMPLE V2 + BASELINES + FOLDS")
print("=" * 80)

# ------------------------------------------------
# 1) CONFIG
# ------------------------------------------------
VOL_TARGET_RAW_COL = TARGET_REGISTRY["vol_regression"]      # target_main_fwd_rv
VOL_VALIDITY_COL = VALIDITY_REGISTRY["vol"]                 # valid_for_vol_main

# log-target settings
VOL_EPS = 1e-8
VOL_TARGET_LOG_COL = "target_main_fwd_rv_log"

# HAR-style engineered baseline columns created here
HAR_FEATURE_COLS = [
    "har_rv_d",
    "har_rv_w",
    "har_rv_m",
    "har_range_d",
    "har_range_w",
    "har_range_m",
    "har_oi_z",
    "har_funding_z"
]

# simple benchmark forecast columns created here
VOL_BENCHMARK_COLS = [
    "vol_bm_last_rv",
    "vol_bm_har_mean",
    "vol_bm_range_proxy"
]

# ------------------------------------------------
# 2) BASIC CHECKS
# ------------------------------------------------
required_cols_for_vol = [
    "timestamp",
    VOL_TARGET_RAW_COL,
    VOL_VALIDITY_COL,
    "eth_rv_6",
    "eth_rv_24",
    "eth_rv_48",
    "eth_range_frac"
]

missing_for_vol = [c for c in required_cols_for_vol if c not in df.columns]
if missing_for_vol:
    raise ValueError(f"Missing required columns for volatility block: {missing_for_vol}")

# ------------------------------------------------
# 3) BUILD LOG-VOL TARGET
# ------------------------------------------------
vol_df = df.copy()

# raw target sanity
if (vol_df[VOL_TARGET_RAW_COL] < 0).any():
    raise ValueError("Volatility target contains negative values, which is invalid.")

vol_df[VOL_TARGET_LOG_COL] = np.log(vol_df[VOL_TARGET_RAW_COL] + VOL_EPS)

print("Raw vol target mean :", float(vol_df[VOL_TARGET_RAW_COL].mean()))
print("Raw vol target std  :", float(vol_df[VOL_TARGET_RAW_COL].std()))
print("Log vol target mean :", float(vol_df[VOL_TARGET_LOG_COL].mean()))
print("Log vol target std  :", float(vol_df[VOL_TARGET_LOG_COL].std()))

# ------------------------------------------------
# 4) BUILD HAR-STYLE PERSISTENCE FEATURES
# ------------------------------------------------
# Daily / weekly / monthly style persistence adapted to your 4h data:
# 1 day = 6 bars, 1 week ≈ 42 bars, 1 month ≈ 126 bars
vol_df["har_rv_d"] = vol_df["eth_rv_6"]
vol_df["har_rv_w"] = vol_df["eth_rv_6"].rolling(42, min_periods=10).mean()
vol_df["har_rv_m"] = vol_df["eth_rv_6"].rolling(126, min_periods=30).mean()

vol_df["har_range_d"] = vol_df["eth_range_frac"].rolling(6, min_periods=3).mean()
vol_df["har_range_w"] = vol_df["eth_range_frac"].rolling(42, min_periods=10).mean()
vol_df["har_range_m"] = vol_df["eth_range_frac"].rolling(126, min_periods=30).mean()

# Ex-ante derivatives state as volatility predictors
if "oi_bybit_z_48" in vol_df.columns:
    vol_df["har_oi_z"] = vol_df["oi_bybit_z_48"]
else:
    vol_df["har_oi_z"] = np.nan

if "funding_rate_z_48" in vol_df.columns:
    vol_df["har_funding_z"] = vol_df["funding_rate_z_48"]
else:
    vol_df["har_funding_z"] = np.nan

# ------------------------------------------------
# 5) BUILD SIMPLE BENCHMARK FORECASTS
# ------------------------------------------------
# These are not the final model; they are reference forecasts
# for the next block's evaluation.
#
# vol_bm_last_rv:
#   persistence benchmark using current RV proxy
#
# vol_bm_har_mean:
#   simple HAR-style average of d/w/m RV components
#
# vol_bm_range_proxy:
#   range-based proxy benchmark
vol_df["vol_bm_last_rv"] = vol_df["eth_rv_6"]
vol_df["vol_bm_har_mean"] = vol_df[["har_rv_d", "har_rv_w", "har_rv_m"]].mean(axis=1)
vol_df["vol_bm_range_proxy"] = vol_df[["har_range_d", "har_range_w", "har_range_m"]].mean(axis=1)

# enforce non-negativity on benchmarks
for c in VOL_BENCHMARK_COLS:
    vol_df[c] = vol_df[c].clip(lower=0)

# ------------------------------------------------
# 6) BUILD VOL FEATURE SET
# ------------------------------------------------
# Start from globally screened numeric candidates, then explicitly
# include HAR-style features and benchmark inputs.
vol_feature_cols = candidate_numeric.copy()
vol_feature_cols = [c for c in vol_feature_cols if c in vol_df.columns]

vol_feature_cols = list(dict.fromkeys(vol_feature_cols + HAR_FEATURE_COLS))
vol_feature_cols = [c for c in vol_feature_cols if c in vol_df.columns]

# remove direct target leakage if present
vol_feature_cols = [
    c for c in vol_feature_cols
    if c not in {VOL_TARGET_RAW_COL, VOL_TARGET_LOG_COL}
]

# ------------------------------------------------
# 7) BUILD CLEAN VOL SAMPLE
# ------------------------------------------------
vol_df = vol_df.loc[vol_df[VOL_VALIDITY_COL] == 1].copy()
vol_df = vol_df.loc[vol_df[VOL_TARGET_RAW_COL].notna()].copy()
vol_df = vol_df.loc[vol_df[VOL_TARGET_LOG_COL].notna()].copy()

before_rows = len(vol_df)

# Require features + target + benchmark availability
required_for_clean = vol_feature_cols + [VOL_TARGET_RAW_COL, VOL_TARGET_LOG_COL] + VOL_BENCHMARK_COLS
required_for_clean = [c for c in required_for_clean if c in vol_df.columns]

vol_df = vol_df.dropna(subset=required_for_clean).copy()
after_rows = len(vol_df)

X_vol = vol_df[vol_feature_cols].copy()
y_vol_raw = vol_df[VOL_TARGET_RAW_COL].astype(float).copy()
y_vol_log = vol_df[VOL_TARGET_LOG_COL].astype(float).copy()

meta_vol = vol_df[["timestamp", VOL_TARGET_RAW_COL, VOL_TARGET_LOG_COL] + VOL_BENCHMARK_COLS + [c for c in FOLD_COLS if c in vol_df.columns]].copy()

print("\n" + "=" * 80)
print("VOL SAMPLE SUMMARY")
print("=" * 80)
print("Rows before dropna:", before_rows)
print("Rows after dropna :", after_rows)
print("Rows removed      :", before_rows - after_rows)
print("X_vol shape       :", X_vol.shape)
print("y_vol_raw shape   :", y_vol_raw.shape)
print("y_vol_log shape   :", y_vol_log.shape)

print("\nTarget summary:")
print("  raw mean:", float(y_vol_raw.mean()))
print("  raw std :", float(y_vol_raw.std()))
print("  log mean:", float(y_vol_log.mean()))
print("  log std :", float(y_vol_log.std()))

print("\nFirst 30 vol features:")
print(vol_feature_cols[:30])

# ------------------------------------------------
# 8) REBUILD VOL FOLDS AFTER CLEANING
# ------------------------------------------------
VOL_FOLD_SPLITS = []

vol_fold_cols = sorted([c for c in vol_df.columns if c in FOLD_COLS])

for fold_col in vol_fold_cols:
    train_idx = vol_df.index[vol_df[fold_col] == "train"].tolist()
    valid_idx = vol_df.index[vol_df[fold_col] == "valid"].tolist()
    test_idx  = vol_df.index[vol_df[fold_col] == "test"].tolist()

    if len(train_idx) == 0 or len(valid_idx) == 0 or len(test_idx) == 0:
        print(f"Skipping {fold_col} because one segment is empty after vol cleaning.")
        continue

    VOL_FOLD_SPLITS.append({
        "fold_name": fold_col,
        "train_idx": train_idx,
        "valid_idx": valid_idx,
        "test_idx": test_idx,
    })

if len(VOL_FOLD_SPLITS) == 0:
    raise ValueError("No usable volatility folds remain after rebuilding cleaned folds.")

# chronology checks
for split in VOL_FOLD_SPLITS:
    fold_name = split["fold_name"]

    train_end = vol_df.loc[split["train_idx"], "timestamp"].max()
    valid_start = vol_df.loc[split["valid_idx"], "timestamp"].min()
    valid_end = vol_df.loc[split["valid_idx"], "timestamp"].max()
    test_start = vol_df.loc[split["test_idx"], "timestamp"].min()

    if not (train_end < valid_start):
        raise ValueError(f"{fold_name} violates train < valid chronology in vol sample")
    if not (valid_end < test_start):
        raise ValueError(f"{fold_name} violates valid < test chronology in vol sample")

# positional maps
row_pos_map_vol = pd.Series(np.arange(len(vol_df)), index=vol_df.index)

for split in VOL_FOLD_SPLITS:
    split["train_pos"] = row_pos_map_vol.loc[split["train_idx"]].tolist()
    split["valid_pos"] = row_pos_map_vol.loc[split["valid_idx"]].tolist()
    split["test_pos"]  = row_pos_map_vol.loc[split["test_idx"]].tolist()

print("\nUsable vol folds:", len(VOL_FOLD_SPLITS))

# ------------------------------------------------
# 9) VOL FOLD SUMMARY
# ------------------------------------------------
vol_fold_rows = []

for split in VOL_FOLD_SPLITS:
    train_y = y_vol_raw.loc[split["train_idx"]]
    valid_y = y_vol_raw.loc[split["valid_idx"]]
    test_y  = y_vol_raw.loc[split["test_idx"]]

    vol_fold_rows.append({
        "fold": split["fold_name"],
        "n_train": len(train_y),
        "n_valid": len(valid_y),
        "n_test": len(test_y),

        "train_mean_raw": float(train_y.mean()),
        "valid_mean_raw": float(valid_y.mean()),
        "test_mean_raw": float(test_y.mean()),

        "train_std_raw": float(train_y.std()),
        "valid_std_raw": float(valid_y.std()),
        "test_std_raw": float(test_y.std()),

        "train_start": vol_df.loc[split["train_idx"], "timestamp"].min(),
        "train_end": vol_df.loc[split["train_idx"], "timestamp"].max(),
        "valid_start": vol_df.loc[split["valid_idx"], "timestamp"].min(),
        "valid_end": vol_df.loc[split["valid_idx"], "timestamp"].max(),
        "test_start": vol_df.loc[split["test_idx"], "timestamp"].min(),
        "test_end": vol_df.loc[split["test_idx"], "timestamp"].max(),
    })

VOL_FOLD_SUMMARY_DF = pd.DataFrame(vol_fold_rows)

print("\n" + "=" * 80)
print("VOL FOLD SUMMARY")
print("=" * 80)
print(VOL_FOLD_SUMMARY_DF)

# ------------------------------------------------
# 10) BENCHMARK SNAPSHOT
# ------------------------------------------------
benchmark_snapshot_rows = []
for col in VOL_BENCHMARK_COLS:
    benchmark_snapshot_rows.append({
        "benchmark": col,
        "mean": float(vol_df[col].mean()),
        "std": float(vol_df[col].std()),
        "corr_with_target_raw": float(vol_df[[col, VOL_TARGET_RAW_COL]].corr().iloc[0, 1]),
        "corr_with_target_log": float(np.corrcoef(vol_df[col], y_vol_log)[0, 1]),
    })

VOL_BENCHMARK_SNAPSHOT_DF = pd.DataFrame(benchmark_snapshot_rows)

print("\n" + "=" * 80)
print("VOL BENCHMARK SNAPSHOT")
print("=" * 80)
print(VOL_BENCHMARK_SNAPSHOT_DF)

# ------------------------------------------------
# 11) SANITY CHECKS
# ------------------------------------------------
if X_vol.empty:
    raise ValueError("X_vol is empty.")

if len(vol_feature_cols) < 15:
    raise ValueError("Too few volatility features remain after filtering.")

if (y_vol_raw < 0).any():
    raise ValueError("Negative realized-vol target detected after cleaning.")

print("\nAll volatility sample sanity checks passed.")

# ------------------------------------------------
# 12) SAVE ARTIFACTS
# ------------------------------------------------
VOL_SAMPLE_PREVIEW_PATH = os.path.join(OUTPUT_DIR, "v2_vol_sample_preview.csv")
VOL_FOLD_SUMMARY_PATH = os.path.join(OUTPUT_DIR, "v2_vol_fold_summary.csv")
VOL_BENCHMARK_SNAPSHOT_PATH = os.path.join(OUTPUT_DIR, "v2_vol_benchmark_snapshot.csv")

vol_df[["timestamp", VOL_TARGET_RAW_COL, VOL_TARGET_LOG_COL] + HAR_FEATURE_COLS + VOL_BENCHMARK_COLS].to_csv(
    VOL_SAMPLE_PREVIEW_PATH, index=False
)
VOL_FOLD_SUMMARY_DF.to_csv(VOL_FOLD_SUMMARY_PATH, index=False)
VOL_BENCHMARK_SNAPSHOT_DF.to_csv(VOL_BENCHMARK_SNAPSHOT_PATH, index=False)

print("\nSaved artifacts:")
print(" ", VOL_SAMPLE_PREVIEW_PATH)
print(" ", VOL_FOLD_SUMMARY_PATH)
print(" ", VOL_BENCHMARK_SNAPSHOT_PATH)

# ------------------------------------------------
# 13) NOTEBOOK GLOBALS FOR NEXT BLOCKS
# ------------------------------------------------
print("\nNotebook globals created:")
print("  VOL_TARGET_RAW_COL")
print("  VOL_TARGET_LOG_COL")
print("  vol_df")
print("  X_vol")
print("  y_vol_raw")
print("  y_vol_log")
print("  meta_vol")
print("  vol_feature_cols")
print("  VOL_FOLD_SPLITS")
print("  VOL_FOLD_SUMMARY_DF")
print("  VOL_BENCHMARK_COLS")
print("  VOL_BENCHMARK_SNAPSHOT_DF")

BLOCK 9 — RESEARCH-GRADE VOLATILITY SAMPLE V2 + BASELINES + FOLDS
Raw vol target mean : 0.022319011286284444
Raw vol target std  : 0.01808519057966552
Log vol target mean : -4.102640388142583
Log vol target std  : 0.813178409735731

VOL SAMPLE SUMMARY
Rows before dropna: 11974
Rows after dropna : 11297
Rows removed      : 677
X_vol shape       : (11297, 98)
y_vol_raw shape   : (11297,)
y_vol_log shape   : (11297,)

Target summary:
  raw mean: 0.02237873005647569
  raw std : 0.01820711749112238
  log mean: -4.102310430860224
  log std : 0.8170405561116447

First 30 vol features:
['eth_open', 'eth_high', 'eth_low', 'eth_close', 'eth_volume', 'eth_quote_volume', 'eth_num_trades', 'eth_taker_buy_volume', 'btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume', 'btc_quote_volume', 'btc_num_trades', 'btc_taker_buy_volume', 'ethbtc_open', 'ethbtc_high', 'ethbtc_low', 'ethbtc_close', 'ethbtc_volume', 'ethbtc_quote_volume', 'ethbtc_num_trades', 'ethbtc_taker_buy_volume', 'funding_rate', 'm

In [14]:
# ============================================================
# NEW STACK NOTEBOOK — BLOCK 10
# TRAIN VOLATILITY MODELS OOF WITH HAR / QLIKE BENCHMARKING
#
# Purpose:
#   1) Train research-grade volatility models on the cleaned vol sample
#   2) Use log-vol target for training
#   3) Evaluate forecasts on raw-vol scale
#   4) Compare models against explicit baselines
#   5) Use QLIKE as the primary ranking metric, with RMSE/MAE secondary
#
# Required globals from previous blocks:
#   - vol_df
#   - X_vol
#   - y_vol_raw
#   - y_vol_log
#   - meta_vol
#   - vol_feature_cols
#   - VOL_FOLD_SPLITS
#   - VOL_BENCHMARK_COLS
#   - HAR_FEATURE_COLS
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("=" * 80)
print("BLOCK 10 — TRAIN VOLATILITY MODELS OOF WITH HAR / QLIKE BENCHMARKING")
print("=" * 80)

# ------------------------------------------------
# 1) HELPERS
# ------------------------------------------------
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def qlike_from_vol(y_true_vol, y_pred_vol, eps=1e-12):
    """
    QLIKE is defined naturally for variance forecasts.
    We convert vol forecasts to variance forecasts:
        realized variance = y_true_vol^2
        forecast variance = y_pred_vol^2
    and compute:
        mean( RV/FV - log(RV/FV) - 1 )
    Lower is better.
    """
    rv = np.square(np.clip(np.asarray(y_true_vol), eps, None))
    fv = np.square(np.clip(np.asarray(y_pred_vol), eps, None))
    ratio = rv / fv
    return float(np.mean(ratio - np.log(ratio) - 1.0))

def safe_positive_vol_from_log(log_vol_pred, min_vol=1e-8):
    return np.clip(np.exp(log_vol_pred), min_vol, None)

def evaluate_vol_forecast(y_true_vol, y_pred_vol):
    y_true_vol = np.asarray(y_true_vol)
    y_pred_vol = np.asarray(y_pred_vol)

    return {
        "rmse": rmse(y_true_vol, y_pred_vol),
        "mae": float(mean_absolute_error(y_true_vol, y_pred_vol)),
        "qlike": qlike_from_vol(y_true_vol, y_pred_vol),
        "corr": float(np.corrcoef(y_true_vol, y_pred_vol)[0, 1]) if len(y_true_vol) > 1 else np.nan,
    }

# ------------------------------------------------
# 2) MODEL DEFINITIONS
# ------------------------------------------------
# Literature-aligned structure:
#   - explicit HAR benchmark model
#   - regularized broad linear model on log vol
#   - broad tree model on log vol
#   - boosted model on log vol
#
# We train on y_vol_log and convert back to raw vol for evaluation.
VOL_MODEL_LIBRARY = {
    "har_linear": {
        "features": [c for c in HAR_FEATURE_COLS if c in X_vol.columns],
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ]),
        "description": "HAR-style linear model on explicit persistence features"
    },
    "broad_elasticnet": {
        "features": vol_feature_cols,
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", ElasticNet(
                alpha=0.001,
                l1_ratio=0.20,
                max_iter=5000,
                random_state=42
            ))
        ]),
        "description": "Broad regularized linear model on all volatility features"
    },
    "broad_tree": {
        "features": vol_feature_cols,
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(
                n_estimators=500,
                max_depth=8,
                min_samples_leaf=20,
                random_state=42,
                n_jobs=-1
            ))
        ]),
        "description": "Broad nonlinear tree model on all volatility features"
    },
    "broad_boost": {
        "features": vol_feature_cols,
        "model": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(
                max_depth=5,
                learning_rate=0.05,
                max_iter=300,
                random_state=42
            ))
        ]),
        "description": "Broad boosted model on all volatility features"
    }
}

print("Volatility models:")
for k, spec in VOL_MODEL_LIBRARY.items():
    print(f"  {k}: {len(spec['features'])} features")

# hard check
for model_name, spec in VOL_MODEL_LIBRARY.items():
    if len(spec["features"]) < 3:
        raise ValueError(f"Model {model_name} has too few usable features.")

# ------------------------------------------------
# 3) OOF STORAGE
# ------------------------------------------------
vol_oof_df = vol_df[["timestamp", VOL_TARGET_RAW_COL, VOL_TARGET_LOG_COL] + VOL_BENCHMARK_COLS].copy()
vol_oof_df = vol_oof_df.rename(columns={
    VOL_TARGET_RAW_COL: "y_true_vol",
    VOL_TARGET_LOG_COL: "y_true_logvol"
})

for model_name in VOL_MODEL_LIBRARY.keys():
    vol_oof_df[f"vol_{model_name}_valid"] = np.nan
    vol_oof_df[f"vol_{model_name}_test"] = np.nan

vol_fold_metric_rows = []

# ------------------------------------------------
# 4) FOLD LOOP
# ------------------------------------------------
for split in VOL_FOLD_SPLITS:
    fold_name = split["fold_name"]

    train_pos = split["train_pos"]
    valid_pos = split["valid_pos"]
    test_pos = split["test_pos"]

    y_train_log = y_vol_log.iloc[train_pos]
    y_valid_raw = y_vol_raw.iloc[valid_pos]
    y_test_raw  = y_vol_raw.iloc[test_pos]

    print("\n" + "-" * 80)
    print(f"Fold: {fold_name}")
    print(f"Train rows: {len(train_pos)} | Valid rows: {len(valid_pos)} | Test rows: {len(test_pos)}")

    # -------------------------
    # Baselines first
    # -------------------------
    for bm_col in VOL_BENCHMARK_COLS:
        valid_pred_bm = vol_df.iloc[valid_pos][bm_col].values
        test_pred_bm  = vol_df.iloc[test_pos][bm_col].values

        valid_metrics = evaluate_vol_forecast(y_valid_raw.values, valid_pred_bm)
        test_metrics  = evaluate_vol_forecast(y_test_raw.values, test_pred_bm)

        vol_fold_metric_rows.append({
            "fold": fold_name,
            "model": bm_col,
            "model_type": "benchmark",
            "n_features": 1,

            "valid_rmse": valid_metrics["rmse"],
            "test_rmse": test_metrics["rmse"],
            "valid_mae": valid_metrics["mae"],
            "test_mae": test_metrics["mae"],
            "valid_qlike": valid_metrics["qlike"],
            "test_qlike": test_metrics["qlike"],
            "valid_corr": valid_metrics["corr"],
            "test_corr": test_metrics["corr"],
        })

    # -------------------------
    # Learned models
    # -------------------------
    for model_name, spec in VOL_MODEL_LIBRARY.items():
        feat_cols = spec["features"]
        model = spec["model"]

        X_train = X_vol.iloc[train_pos][feat_cols]
        X_valid = X_vol.iloc[valid_pos][feat_cols]
        X_test  = X_vol.iloc[test_pos][feat_cols]

        model.fit(X_train, y_train_log)

        valid_pred_log = model.predict(X_valid)
        test_pred_log  = model.predict(X_test)

        valid_pred_vol = safe_positive_vol_from_log(valid_pred_log)
        test_pred_vol  = safe_positive_vol_from_log(test_pred_log)

        vol_oof_df.loc[X_valid.index, f"vol_{model_name}_valid"] = valid_pred_vol
        vol_oof_df.loc[X_test.index, f"vol_{model_name}_test"] = test_pred_vol

        valid_metrics = evaluate_vol_forecast(y_valid_raw.values, valid_pred_vol)
        test_metrics  = evaluate_vol_forecast(y_test_raw.values, test_pred_vol)

        vol_fold_metric_rows.append({
            "fold": fold_name,
            "model": model_name,
            "model_type": "learned",
            "n_features": len(feat_cols),

            "valid_rmse": valid_metrics["rmse"],
            "test_rmse": test_metrics["rmse"],
            "valid_mae": valid_metrics["mae"],
            "test_mae": test_metrics["mae"],
            "valid_qlike": valid_metrics["qlike"],
            "test_qlike": test_metrics["qlike"],
            "valid_corr": valid_metrics["corr"],
            "test_corr": test_metrics["corr"],
        })

        print(
            f"{model_name:>16} | "
            f"valid_qlike={valid_metrics['qlike']:.6f} | "
            f"test_qlike={test_metrics['qlike']:.6f} | "
            f"valid_rmse={valid_metrics['rmse']:.6f}"
        )

# ------------------------------------------------
# 5) METRIC TABLES
# ------------------------------------------------
VOL_FOLD_METRICS_DF = pd.DataFrame(vol_fold_metric_rows)

print("\n" + "=" * 80)
print("VOL FOLD METRICS (HEAD)")
print("=" * 80)
print(VOL_FOLD_METRICS_DF.head())

VOL_METRICS_SUMMARY_DF = (
    VOL_FOLD_METRICS_DF
    .groupby(["model", "model_type"])[
        ["valid_qlike", "test_qlike", "valid_rmse", "test_rmse",
         "valid_mae", "test_mae", "valid_corr", "test_corr"]
    ]
    .agg(["mean", "std"])
    .sort_values(("valid_qlike", "mean"), ascending=True)  # lower QLIKE is better
)

print("\n" + "=" * 80)
print("VOL METRICS SUMMARY (RANKED BY VALID QLIKE)")
print("=" * 80)
print(VOL_METRICS_SUMMARY_DF)

# ------------------------------------------------
# 6) OOF COVERAGE CHECK
# ------------------------------------------------
coverage_rows = []
for model_name in VOL_MODEL_LIBRARY.keys():
    coverage_rows.append({
        "model": model_name,
        "valid_nonnull": int(vol_oof_df[f"vol_{model_name}_valid"].notna().sum()),
        "test_nonnull": int(vol_oof_df[f"vol_{model_name}_test"].notna().sum()),
    })

VOL_OOF_COVERAGE_DF = pd.DataFrame(coverage_rows)

print("\n" + "=" * 80)
print("VOL OOF COVERAGE CHECK")
print("=" * 80)
print(VOL_OOF_COVERAGE_DF)

# ------------------------------------------------
# 7) META-READY VOL FRAME
# ------------------------------------------------
vol_model_valid_cols = [f"vol_{m}_valid" for m in VOL_MODEL_LIBRARY.keys()]

vol_meta_df = vol_oof_df[["timestamp", "y_true_vol", "y_true_logvol"] + VOL_BENCHMARK_COLS + vol_model_valid_cols].copy()
vol_meta_df = vol_meta_df.dropna(subset=vol_model_valid_cols).copy()

print("\n" + "=" * 80)
print("VOL META DATASET")
print("=" * 80)
print("vol_meta_df shape:", vol_meta_df.shape)

# ------------------------------------------------
# 8) SIMPLE MODEL COMPARISON ON META SAMPLE
# ------------------------------------------------
vol_compare_rows = []

# Benchmarks
for bm_col in VOL_BENCHMARK_COLS:
    metrics = evaluate_vol_forecast(vol_meta_df["y_true_vol"].values, vol_meta_df[bm_col].values)
    vol_compare_rows.append({
        "model": bm_col,
        "model_type": "benchmark",
        "rmse": metrics["rmse"],
        "mae": metrics["mae"],
        "qlike": metrics["qlike"],
        "corr": metrics["corr"],
    })

# Learned models
for model_name in VOL_MODEL_LIBRARY.keys():
    col = f"vol_{model_name}_valid"
    metrics = evaluate_vol_forecast(vol_meta_df["y_true_vol"].values, vol_meta_df[col].values)
    vol_compare_rows.append({
        "model": model_name,
        "model_type": "learned",
        "rmse": metrics["rmse"],
        "mae": metrics["mae"],
        "qlike": metrics["qlike"],
        "corr": metrics["corr"],
    })

VOL_COMPARE_DF = pd.DataFrame(vol_compare_rows).sort_values("qlike", ascending=True).reset_index(drop=True)

print("\n" + "=" * 80)
print("VOL MODEL VS BENCHMARK COMPARISON")
print("=" * 80)
print(VOL_COMPARE_DF)

# ------------------------------------------------
# 9) SAVE ARTIFACTS
# ------------------------------------------------
VOL_OOF_PATH = os.path.join(OUTPUT_DIR, "v2_vol_oof_predictions.csv")
VOL_FOLD_METRICS_PATH = os.path.join(OUTPUT_DIR, "v2_vol_fold_metrics.csv")
VOL_COMPARE_PATH = os.path.join(OUTPUT_DIR, "v2_vol_compare.csv")

vol_oof_df.to_csv(VOL_OOF_PATH, index=False)
VOL_FOLD_METRICS_DF.to_csv(VOL_FOLD_METRICS_PATH, index=False)
VOL_COMPARE_DF.to_csv(VOL_COMPARE_PATH, index=False)

print("\nSaved artifacts:")
print(" ", VOL_OOF_PATH)
print(" ", VOL_FOLD_METRICS_PATH)
print(" ", VOL_COMPARE_PATH)

# ------------------------------------------------
# 10) NOTEBOOK GLOBALS FOR NEXT BLOCKS
# ------------------------------------------------
print("\nNotebook globals created:")
print("  VOL_MODEL_LIBRARY")
print("  vol_oof_df")
print("  VOL_FOLD_METRICS_DF")
print("  VOL_METRICS_SUMMARY_DF")
print("  VOL_OOF_COVERAGE_DF")
print("  vol_meta_df")
print("  VOL_COMPARE_DF")

BLOCK 10 — TRAIN VOLATILITY MODELS OOF WITH HAR / QLIKE BENCHMARKING
Volatility models:
  har_linear: 8 features
  broad_elasticnet: 98 features
  broad_tree: 98 features
  broad_boost: 98 features

--------------------------------------------------------------------------------
Fold: wf_fold_0
Train rows: 4145 | Valid rows: 607 | Test rows: 705
      har_linear | valid_qlike=1.399559 | test_qlike=1.142936 | valid_rmse=0.015877
broad_elasticnet | valid_qlike=2.968095 | test_qlike=4.318970 | valid_rmse=0.015894
      broad_tree | valid_qlike=1.324168 | test_qlike=1.060217 | valid_rmse=0.014828
     broad_boost | valid_qlike=2.586522 | test_qlike=1.533946 | valid_rmse=0.015858

--------------------------------------------------------------------------------
Fold: wf_fold_1
Train rows: 4084 | Valid rows: 705 | Test rows: 720
      har_linear | valid_qlike=1.341287 | test_qlike=1.140765 | valid_rmse=0.011917
broad_elasticnet | valid_qlike=1.327130 | test_qlike=2.416338 | valid_rmse=0.01144

/Users/carlosrubiano/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.484e+00, tolerance: 3.179e-01
  model = cd_fast.enet_coordinate_descent(


broad_elasticnet | valid_qlike=0.957276 | test_qlike=1.157303 | valid_rmse=0.015458
      broad_tree | valid_qlike=1.236327 | test_qlike=0.921926 | valid_rmse=0.014588
     broad_boost | valid_qlike=1.404564 | test_qlike=1.055782 | valid_rmse=0.014953

--------------------------------------------------------------------------------
Fold: wf_fold_5
Train rows: 4059 | Valid rows: 694 | Test rows: 628
      har_linear | valid_qlike=1.719598 | test_qlike=1.905595 | valid_rmse=0.015073


/Users/carlosrubiano/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.311e+00, tolerance: 3.061e-01
  model = cd_fast.enet_coordinate_descent(


broad_elasticnet | valid_qlike=1.576675 | test_qlike=0.898169 | valid_rmse=0.013950
      broad_tree | valid_qlike=0.894905 | test_qlike=0.964574 | valid_rmse=0.012755
     broad_boost | valid_qlike=1.103481 | test_qlike=1.175278 | valid_rmse=0.013099

--------------------------------------------------------------------------------
Fold: wf_fold_6
Train rows: 4033 | Valid rows: 628 | Test rows: 720
      har_linear | valid_qlike=1.666977 | test_qlike=1.702105 | valid_rmse=0.019061
broad_elasticnet | valid_qlike=0.815886 | test_qlike=0.964708 | valid_rmse=0.022143
      broad_tree | valid_qlike=0.921200 | test_qlike=0.982316 | valid_rmse=0.015616
     broad_boost | valid_qlike=1.204933 | test_qlike=1.182907 | valid_rmse=0.016377

--------------------------------------------------------------------------------
Fold: wf_fold_7
Train rows: 4028 | Valid rows: 720 | Test rows: 689
      har_linear | valid_qlike=1.463781 | test_qlike=1.324047 | valid_rmse=0.018489
broad_elasticnet | valid_qli

/Users/carlosrubiano/Library/Python/3.9/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:678: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.425e+00, tolerance: 2.649e-01
  model = cd_fast.enet_coordinate_descent(


broad_elasticnet | valid_qlike=0.796304 | test_qlike=0.978719 | valid_rmse=0.012087
      broad_tree | valid_qlike=0.686327 | test_qlike=0.898322 | valid_rmse=0.010636
     broad_boost | valid_qlike=0.685113 | test_qlike=0.848959 | valid_rmse=0.011231

VOL FOLD METRICS (HEAD)
        fold               model model_type  n_features  valid_rmse  \
0  wf_fold_0      vol_bm_last_rv  benchmark           1    0.017076   
1  wf_fold_0     vol_bm_har_mean  benchmark           1    0.017641   
2  wf_fold_0  vol_bm_range_proxy  benchmark           1    0.015832   
3  wf_fold_0          har_linear    learned           8    0.015877   
4  wf_fold_0    broad_elasticnet    learned          98    0.015894   

   test_rmse  valid_mae  test_mae  valid_qlike  test_qlike  valid_corr  \
0   0.013457   0.010736  0.009247     9.264010    6.005151    0.454258   
1   0.012984   0.010366  0.008389     4.061920    3.031123    0.409192   
2   0.011912   0.011555  0.008819     1.353023    1.198874    0.440562   


In [15]:
# ============================================================
# NEW STACK NOTEBOOK — BLOCK 11
# FINAL V2 SIGNAL INTEGRATION ARTIFACT
#
# Purpose:
#   1) Merge the refined alpha stack with the best volatility forecast
#   2) Use alpha confidence as the practical gate proxy
#   3) Build a clean deployment-style signal artifact
#   4) Compute final score, risk scaling, and desired position
#   5) Save everything for the separate backtest notebook
#
# Design:
#   - Alpha signal    : alpha_stack_prob
#   - Gate proxy      : alpha_confidence = max(p, 1-p)
#   - Vol forecast    : vol_broad_tree_valid
#   - No learned gate in v2 (it underperformed the baseline)
# ============================================================

print("=" * 80)
print("BLOCK 11 — FINAL V2 SIGNAL INTEGRATION ARTIFACT")
print("=" * 80)

# ------------------------------------------------
# 1) REQUIRED INPUT CHECKS
# ------------------------------------------------
required_alpha_cols = ["timestamp", "alpha_stack_prob", "y_true"]
missing_alpha = [c for c in required_alpha_cols if c not in alpha_stack_df.columns]
if missing_alpha:
    raise ValueError(f"Missing required alpha stack columns: {missing_alpha}")

required_vol_cols = ["timestamp", "y_true_vol", "vol_broad_tree_valid"]
missing_vol = [c for c in required_vol_cols if c not in vol_meta_df.columns]
if missing_vol:
    raise ValueError(f"Missing required vol meta columns: {missing_vol}")

if "target_main_ret" not in df.columns:
    raise ValueError("Missing target_main_ret in df.")

# ------------------------------------------------
# 2) BUILD BASE INTEGRATION FRAME
# ------------------------------------------------
integration_df = alpha_stack_df[["timestamp", "y_true", "alpha_stack_prob"]].copy()

# alpha confidence as gate proxy
integration_df["alpha_confidence"] = np.maximum(
    integration_df["alpha_stack_prob"],
    1.0 - integration_df["alpha_stack_prob"]
)
integration_df["alpha_side"] = np.where(integration_df["alpha_stack_prob"] >= 0.5, 1, -1)
integration_df["alpha_edge"] = (integration_df["alpha_stack_prob"] - 0.5) * 2.0
integration_df["alpha_abs_edge"] = integration_df["alpha_edge"].abs()

# merge best vol forecast
integration_df = integration_df.merge(
    vol_meta_df[["timestamp", "y_true_vol", "vol_broad_tree_valid"]].copy(),
    on="timestamp",
    how="inner"
)

# merge realized forward return for later external evaluation / backtest notebook
integration_df = integration_df.merge(
    df[["timestamp", "target_main_ret"]].copy(),
    on="timestamp",
    how="left"
)

integration_df = integration_df.sort_values("timestamp").reset_index(drop=True)

if integration_df.empty:
    raise ValueError("integration_df is empty after alpha/vol merge.")

print("integration_df shape:", integration_df.shape)

# ------------------------------------------------
# 3) RISK SCALING FROM VOL FORECAST
# ------------------------------------------------
# Use broad_tree vol forecast as the main ex-ante risk estimate
integration_df["vol_forecast"] = integration_df["vol_broad_tree_valid"].copy()

# safety floor and normalization
vol_floor = integration_df["vol_forecast"].quantile(0.10)
integration_df["vol_used"] = np.maximum(integration_df["vol_forecast"], vol_floor)

integration_df["inv_vol_scale_raw"] = 1.0 / integration_df["vol_used"]
integration_df["inv_vol_scale"] = (
    integration_df["inv_vol_scale_raw"] /
    integration_df["inv_vol_scale_raw"].median()
)

# clip extreme sizes for stability
integration_df["inv_vol_scale"] = integration_df["inv_vol_scale"].clip(0.5, 1.5)

# ------------------------------------------------
# 4) FINAL SCORE CONSTRUCTION
# ------------------------------------------------
# v2 architecture:
#   alpha edge × confidence proxy × inverse-vol scaling
integration_df["confidence_weight"] = integration_df["alpha_confidence"]
integration_df["final_score_raw"] = (
    integration_df["alpha_edge"] *
    integration_df["confidence_weight"] *
    integration_df["inv_vol_scale"]
)

# optional soft clip to prevent extreme values
integration_df["final_score"] = integration_df["final_score_raw"].clip(-1.0, 1.0)

# ------------------------------------------------
# 5) POSITION MAPPING
# ------------------------------------------------
# This is NOT a backtest. It is only the deployment-style signal artifact.
# The backtest notebook can choose to use these defaults or tune them.
ENTRY_THRESHOLD = 0.20
NEUTRAL_BAND = 0.10
MAX_ABS_POSITION = 1.0

integration_df["desired_position"] = 0.0

long_mask = integration_df["final_score"] > ENTRY_THRESHOLD
short_mask = integration_df["final_score"] < -ENTRY_THRESHOLD
neutral_mask = integration_df["final_score"].abs() < NEUTRAL_BAND

integration_df.loc[long_mask, "desired_position"] = (
    integration_df.loc[long_mask, "final_score"].clip(upper=MAX_ABS_POSITION)
)
integration_df.loc[short_mask, "desired_position"] = (
    integration_df.loc[short_mask, "final_score"].clip(lower=-MAX_ABS_POSITION)
)
integration_df.loc[neutral_mask, "desired_position"] = 0.0

# ------------------------------------------------
# 6) SIGNAL DIAGNOSTICS
# ------------------------------------------------
integration_df["score_bucket"] = pd.cut(
    integration_df["final_score"],
    bins=[-np.inf, -0.50, -0.20, -0.10, 0.10, 0.20, 0.50, np.inf],
    labels=["<=-0.50", "(-0.50,-0.20]", "(-0.20,-0.10]", "neutral", "(0.10,0.20]", "(0.20,0.50]", ">0.50"]
)

signal_summary = {
    "n_rows": len(integration_df),
    "start": integration_df["timestamp"].min(),
    "end": integration_df["timestamp"].max(),

    "alpha_prob_mean": float(integration_df["alpha_stack_prob"].mean()),
    "alpha_confidence_mean": float(integration_df["alpha_confidence"].mean()),
    "vol_forecast_mean": float(integration_df["vol_forecast"].mean()),
    "inv_vol_scale_mean": float(integration_df["inv_vol_scale"].mean()),

    "final_score_mean": float(integration_df["final_score"].mean()),
    "final_score_std": float(integration_df["final_score"].std()),

    "fraction_nonzero_desired_position": float((integration_df["desired_position"] != 0).mean()),
    "fraction_long_desired": float((integration_df["desired_position"] > 0).mean()),
    "fraction_short_desired": float((integration_df["desired_position"] < 0).mean()),
    "avg_abs_desired_position": float(integration_df["desired_position"].abs().mean()),
    "max_abs_desired_position": float(integration_df["desired_position"].abs().max()),
}

INTEGRATION_SUMMARY_DF = pd.DataFrame({
    "metric": list(signal_summary.keys()),
    "value": list(signal_summary.values())
})

print("\n" + "=" * 80)
print("INTEGRATION SUMMARY")
print("=" * 80)
for _, row in INTEGRATION_SUMMARY_DF.iterrows():
    print(f"{row['metric']}: {row['value']}")

print("\n" + "=" * 80)
print("SCORE BUCKET COUNTS")
print("=" * 80)
print(integration_df["score_bucket"].value_counts(dropna=False).sort_index())

# ------------------------------------------------
# 7) FEATURE / SIGNAL PREVIEW
# ------------------------------------------------
preview_cols = [
    "timestamp",
    "y_true",
    "target_main_ret",
    "alpha_stack_prob",
    "alpha_confidence",
    "alpha_side",
    "alpha_edge",
    "vol_forecast",
    "inv_vol_scale",
    "final_score",
    "desired_position"
]

print("\n" + "=" * 80)
print("INTEGRATED SIGNAL PREVIEW")
print("=" * 80)
print(integration_df[preview_cols].head())

# ------------------------------------------------
# 8) SAVE ARTIFACTS
# ------------------------------------------------
INTEGRATION_PATH = os.path.join(OUTPUT_DIR, "v2_final_signal_integration.csv")
INTEGRATION_SUMMARY_PATH = os.path.join(OUTPUT_DIR, "v2_final_signal_integration_summary.csv")

integration_df.to_csv(INTEGRATION_PATH, index=False)
INTEGRATION_SUMMARY_DF.to_csv(INTEGRATION_SUMMARY_PATH, index=False)

print("\nSaved artifacts:")
print(" ", INTEGRATION_PATH)
print(" ", INTEGRATION_SUMMARY_PATH)

# ------------------------------------------------
# 9) NOTEBOOK GLOBALS FOR LATER NOTEBOOKS
# ------------------------------------------------
print("\nNotebook globals created:")
print("  integration_df")
print("  INTEGRATION_SUMMARY_DF")
print("  INTEGRATION_PATH")

BLOCK 11 — FINAL V2 SIGNAL INTEGRATION ARTIFACT
integration_df shape: (6010, 10)

INTEGRATION SUMMARY
n_rows: 6010
start: 2022-10-21 00:00:00+00:00
end: 2025-10-04 20:00:00+00:00
alpha_prob_mean: 0.5119163328333372
alpha_confidence_mean: 0.7302529491523139
vol_forecast_mean: 0.01473834749227099
inv_vol_scale_mean: 1.026676882820852
final_score_mean: 0.011347991219189195
final_score_std: 0.4204700998842957
fraction_nonzero_desired_position: 0.5735440931780366
fraction_long_desired: 0.29700499168053246
fraction_short_desired: 0.27653910149750416
avg_abs_desired_position: 0.29667619059622324
max_abs_desired_position: 1.0

SCORE BUCKET COUNTS
score_bucket
<=-0.50           825
(-0.50,-0.20]     837
(-0.20,-0.10]     451
neutral          1620
(0.10,0.20]       492
(0.20,0.50]       887
>0.50             898
Name: count, dtype: int64

INTEGRATED SIGNAL PREVIEW
                  timestamp  y_true  target_main_ret  alpha_stack_prob  \
0 2022-10-21 00:00:00+00:00     1.0         0.007794       